# Operationalising Selected Components of CEDAR: A Computational Case Study on Scientific Claim Verification with SciFact

**Companion notebook to the paper:** *Detecting Misinterpreted but Genuine Citations Through Discourse-Scoped Claim Decomposition, Multimodal Evidence Retrieval, and Conformal Epistemic Verification* (T. Kumar, IIIT Allahabad).

> **Status of this notebook (please read).**
> The paper is a literature-based methodological framework (CEDAR). It reports **no executed experiments** (paper, Sec. VIII note and Sec. IX-A.4). This notebook is an **experimental computational case study** that operationalises *selected* CEDAR components on an existing public benchmark (SciFact). It does **not** validate the full 12-stage framework, and it does **not** prove that CEDAR is effective.
> **No result is stored in this notebook.** Every number, table and figure is produced when the cells are executed. Sections 15 and 17 contain clearly marked spaces for the author's own interpretation of the numbers they obtain.

### Provenance legend used throughout

| Tag | Meaning |
|---|---|
| **[PAPER]** | Concept, equation or stage taken from the CEDAR paper |
| **[DATASET]** | Existing public benchmark data (SciFact) |
| **[MODEL]** | Existing pretrained model, used as-is (or fine-tuned where stated) |
| **[IMPL]** | Method implemented in this notebook (simplified relative to the paper where stated) |
| **[OBS]** | Original observation, computed at execution time from real benchmark data |
| **[ILLUS]** | Manually constructed illustrative example; **never** included in any metric |

## 0. Setup (run first)

* **Internet access is required** for `pip install`, for downloading SciFact, and for downloading Hugging Face models.
* Recommended runtime: Google Colab. A GPU is *optional*; the default configuration runs on CPU but the dense retriever and NLI scoring are faster with a GPU.
* All behaviour is controlled from the `CONFIG` dictionary in the next cell (seeds, retrieval depths `K`, optional components, bootstrap size).

In [ ]:
# Installation (Google Colab). Colab already ships numpy, pandas, scikit-learn, scipy, matplotlib,
# seaborn, torch and transformers; only the packages below are typically missing.
import importlib, subprocess, sys

def ensure_package(import_name, pip_name):
    """Install a package with pip if it cannot be imported; warn (do not crash) if installation fails."""
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing {pip_name} ...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        except Exception as exc:  # network problems etc.
            print(f"WARNING: could not install {pip_name}: {exc}")

for _imp, _pip in [("rank_bm25", "rank-bm25"),
                   ("sentence_transformers", "sentence-transformers"),
                   ("sentencepiece", "sentencepiece")]:
    ensure_package(_imp, _pip)

In [ ]:
import os, re, json, math, random, platform, tarfile, unicodedata, urllib.request, warnings
from pathlib import Path
from collections import Counter
from importlib import metadata as importlib_metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse, stats as sps
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

try:
    from IPython.display import display
except ImportError:  # plain-python fallback
    display = print

try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

sns.set_theme(style="whitegrid", context="notebook")

# ----------------------------------------------------------------------------------------------
# Single place that controls the experiment. Nothing below hard-codes a result.
# ----------------------------------------------------------------------------------------------
CONFIG = {
    "SEED": 42,
    "DATA_URL": "https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz",
    "DATA_DIR": "data/scifact",
    "RESULTS_DIR": "results",
    "FIG_DIR": "results/figures",
    # --- data partitioning (official train pool -> train / validation / conformal-calibration) ---
    "TRAIN_FRACTION": 0.70,            # remaining 30% is split equally into validation and calibration
    "DROP_TEST_DOC_OVERLAP": True,     # drop pool claims whose cited documents are cited by test claims
    # --- retrieval ---
    "K_SENT": [1, 3, 5],               # evidence depths for sentence retrieval / verification
    "K_DOC": [1, 5, 10, 20],           # cut-offs for corpus-level abstract retrieval
    "USE_DENSE": True,                 # dense retriever (sentence-transformers); needs model download
    "DENSE_MODEL": "sentence-transformers/all-MiniLM-L6-v2",
    # --- verification ---
    "USE_NLI": True,                   # pretrained NLI model; needs model download
    "NLI_MODEL": "cross-encoder/nli-deberta-v3-small",
    "NLI_BATCH_SIZE": 32,
    "NLI_MAX_LEN": 256,
    "RUN_FINETUNE": False,             # OPTIONAL advanced component (GPU strongly recommended)
    "FT_EPOCHS": 2, "FT_LR": 2e-5, "FT_BATCH": 8,
    # --- statistics / uncertainty ---
    "N_BOOT": 2000,                    # cluster-bootstrap replicates
    "CONF_LEVEL": 0.95,
    "CONFORMAL_ALPHA": 0.10,
    "ECE_BINS": 10,
    "NUM_TOL": 0.05,                   # tolerance for the crude numeric-consistency proxy (paper: eps_tol, unspecified)
}

def set_seed(seed):
    """Seed every random number generator we use."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if HAS_TORCH:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["SEED"])

RESULTS_DIR = Path(CONFIG["RESULTS_DIR"]); FIG_DIR = Path(CONFIG["FIG_DIR"])
RESULTS_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(parents=True, exist_ok=True)

def savefig(name):
    """Save the current matplotlib figure to results/figures and show it."""
    plt.tight_layout()
    plt.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")
    plt.show()

def _version(pkg):
    try:
        return importlib_metadata.version(pkg)
    except Exception:
        return "not installed"

ENVIRONMENT = {"python": platform.python_version(), "platform": platform.platform(),
               **{p: _version(p) for p in ["numpy", "pandas", "scikit-learn", "scipy", "matplotlib", "seaborn",
                                           "rank-bm25", "sentence-transformers", "transformers", "torch"]}}
print(json.dumps(ENVIRONMENT, indent=2))

In [ ]:
# ---------------------------- shared statistical / metric utilities ----------------------------
# (the metrics themselves are explained in Section 11; they are defined here so that every later
#  section can use the same, consistent implementation)

def confusion_np(y, p, k=3):
    """k x k confusion matrix, rows = true class, columns = predicted class."""
    y = np.asarray(y, dtype=int); p = np.asarray(p, dtype=int)
    return np.bincount(y * k + p, minlength=k * k).reshape(k, k)

def macro_f1_np(y, p, k=3):
    """Macro-F1 over all k classes (a class with no true and no predicted items contributes 0, like sklearn's zero_division=0)."""
    cm = confusion_np(y, p, k)
    tp = np.diag(cm).astype(float); pred = cm.sum(0).astype(float); true = cm.sum(1).astype(float)
    prec = np.divide(tp, pred, out=np.zeros(k), where=pred > 0)
    rec = np.divide(tp, true, out=np.zeros(k), where=true > 0)
    den = prec + rec
    f1 = np.divide(2 * prec * rec, den, out=np.zeros(k), where=den > 0)
    return float(f1.mean())

def accuracy_np(y, p):
    return float(np.mean(np.asarray(y) == np.asarray(p)))

def _group_members(groups):
    groups = np.asarray(groups)
    _, inv = np.unique(groups, return_inverse=True)
    return [np.flatnonzero(inv == g) for g in range(int(inv.max()) + 1)]

def cluster_bootstrap(stat_fn, groups, n_boot=None, conf=None, seed=None):
    """Percentile bootstrap CI that resamples whole *groups* (connected components of cited documents).

    Claims that cite the same paper are not independent; resampling groups respects that dependence.
    stat_fn(idx) must return a float computed on the claims selected by the integer index array idx.
    Returns (point_estimate, ci_low, ci_high)."""
    n_boot = n_boot or CONFIG["N_BOOT"]; conf = conf or CONFIG["CONF_LEVEL"]
    rng = np.random.default_rng(CONFIG["SEED"] if seed is None else seed)
    members = _group_members(groups); G = len(members)
    point = float(stat_fn(np.arange(len(groups))))
    vals = np.empty(n_boot)
    for b in range(n_boot):
        draw = rng.integers(0, G, G)
        vals[b] = stat_fn(np.concatenate([members[g] for g in draw]))
    a = (1 - conf) / 2
    return point, float(np.nanquantile(vals, a)), float(np.nanquantile(vals, 1 - a))

def mcnemar_exact(correct_a, correct_b):
    """Exact two-sided McNemar test on paired correctness vectors. Returns (b, c, p) where
    b = #(A right, B wrong), c = #(A wrong, B right)."""
    ca = np.asarray(correct_a, dtype=bool); cb = np.asarray(correct_b, dtype=bool)
    b = int(np.sum(ca & ~cb)); c = int(np.sum(~ca & cb)); n = b + c
    p = 1.0 if n == 0 else float(sps.binomtest(min(b, c), n, 0.5).pvalue)
    return b, c, p

def holm_adjust(pvals):
    """Holm-Bonferroni step-down adjusted p-values."""
    p = np.asarray(pvals, dtype=float); m = len(p)
    order = np.argsort(p); adj = np.empty(m); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i]); adj[i] = min(1.0, running)
    return adj

## 1. Research Background and Objective

**Problem [PAPER].** A *misinterpreted but genuine citation* points to a real, correctly catalogued publication, yet the citing sentence exaggerates, over-generalises, inflates correlation into causation, distorts numbers, or contradicts what the cited work reports. Existence checks (DOI resolution) and topical-similarity tools cannot detect this; detecting it requires checking whether the **cited paper's content entails the citing claim** (paper, Table I, Layers 4-5).

**Objective of this notebook.** Build and evaluate an automated pipeline that, given a scientific claim and its cited source, retrieves evidence from the source and decides whether the evidence **supports**, **refutes**, or provides **not enough information (NEI)** for the claim, and study how the choice of *retrieval method* and *verification model* affects quality.

**Scope.** This is a three-label proxy for the paper's four-way label space (Supported / Partially Supported / Contradicted / Neutral). SciFact has no "partially supported" label, so that class cannot be evaluated here.

## 2. Relationship Between the Research Paper and the Experiment

| Notebook component | CEDAR stage / equation **[PAPER]** | Status in this notebook |
|---|---|---|
| Known cited document per claim; bibliographic validity assumed | Stage 5 (authenticity, horizon flag `h_doc`) | **Assumed**: SciFact abstracts are genuine and fully accessible; no DOI resolution is performed |
| Light claim normalisation and heuristic splitting of compound claims | Stages 2-4 (discourse window, atomic decomposition, bipartite claim-anchor map) | **Simplified rule-based demonstration** (Sec. 6). No coreference resolution, no discourse window, no LLM. Not evaluated against gold atomic claims |
| TF-IDF, BM25 (and optional dense) sentence retrieval | Stage 7 (BM25 + dense; Eqs. 7-8) | **Implemented** at the sentence level. ColBERT, table/figure indexing and section weighting (Eqs. 9-10) are **not** implemented |
| Retrieval quality: Recall@K, MRR | Table VI, dimension 2 | **Implemented** |
| Pretrained NLI on (evidence, claim) | Stage 9 | **Implemented** (general-domain NLI, zero-shot; optional fine-tuning) |
| Relevance gate + confidence gate + abstention | Stage 11 routing idea (`|C_alpha| >= 2` -> human review), Stage 12 | **Implemented as a simple threshold rule** (Sec. 9), *not* the paper's full composite score (Eq. 16) |
| Temperature scaling, ECE, Brier; split conformal prediction sets | Stage 11, Eqs. 14-15 | **Implemented** on the NLI posteriors (Sec. 12) |
| Cue-based error categories | Taxonomy T1-T8 (Table III) | **Heuristic lexical proxies** (Sec. 14); *not* the paper's modality regressor, scope frames, or human annotation |
| Crude numeric consistency check | Eq. 13 (`delta_num`) | **Simplified** nearest-number alignment (Sec. 14) |
| Epistemic Modality Delta (Eq. 11), scope divergence (Eq. 12), cross-encoder reranking (Stage 8), multimodal evidence, citation-graph splitting beyond connected components | Stages 6, 8, 10 | **Not implemented** |
| Evaluation protocol: leakage-free splits, paired bootstrap, McNemar + Holm | Sec. VIII-A.3, VIII-B | **Implemented** (cluster bootstrap by cited-document component) |

## 3. Research Questions and Experimental Hypotheses

Hypotheses are stated **before** any result is computed. They are directional expectations to be checked, not claimed outcomes; Section 15 evaluates them mechanically against the data.

* **RQ-A (retrieval).** How do TF-IDF, BM25 and (optionally) dense retrieval differ in finding gold rationale sentences inside the cited abstract, and gold abstracts in the full corpus?
  * **H1.** BM25 achieves a sentence-level MRR at least as high as TF-IDF (expected difference: small).
  * **H2.** The dense retriever's abstract-level Recall@K differs from the sparse retrievers' (direction not pre-specified).
* **RQ-B (verification).** Does a pretrained NLI model distinguish support / refutation / insufficient evidence better than a TF-IDF classifier or a similarity-only heuristic?
  * **H3.** A retrieval-only similarity heuristic cannot detect refutation (it can only separate "similar" from "not similar") and therefore has lower macro-F1 than NLI-based verification. *This operationalises the paper's "topical overlap trap".*
  * **H4.** Evidence depth `K` and retriever choice change verification quality (direction not pre-specified).
* **RQ-C (uncertainty).** Do relevance and confidence gates with abstention, temperature scaling and split-conformal sets give usable uncertainty handling on top of NLI?
  * **H5.** Raw NLI posteriors are not perfectly calibrated (ECE > 0); temperature scaling fitted on validation data changes ECE on test data.
* **RQ-D (errors).** Do verification errors concentrate in claims carrying lexical cues related to the paper's taxonomy (numbers, causal verbs, universal quantifiers, scope shifts, hedging erasure, compound claims)?
  * **H6.** Error rates differ between cue categories. *Cues are heuristic proxies, so this is exploratory.*

## 4. Dataset Description and Acquisition

**SciFact [DATASET]** (Wadden et al., EMNLP 2020; <https://github.com/allenai/scifact>). Expert-written scientific claims paired with research abstracts. Each claim is annotated as **SUPPORT**, **CONTRADICT**, or has **no evidence** (treated here as **NOT_ENOUGH_INFO**); labelled claims also carry the **rationale sentences** inside the evidence abstract(s). Each claim lists its `cited_doc_ids`.

**Why it is relevant.** According to the SciFact paper, claims were written from citation sentences in biomedical articles, so a claim paired with its cited abstract is structurally similar to the "citing assertion vs cited source" setting of the paper. The task "does the cited abstract support the claim?" is the Layer-4 (propositional entailment) question of Table I.

**Why it is only a proxy (limitations).**
* SciFact is **not** a benchmark of *misinterpreted* citations. Its REFUTED claims are, to my understanding of the SciFact paper, largely created by annotators negating or altering claims, not naturally occurring misinterpretations; verify this in the SciFact paper before citing it. Its NEI claims mean "this abstract holds no relevant evidence", not "the citation is misleading".
* Evidence comes from **abstracts only**; the paper's concern with full text, tables and figures is outside this data.
* It is biomedical-only; the paper discusses ML and materials science as well.
* The paper's taxonomy T1-T8 is **not annotated** in SciFact.

**Licence.** To my knowledge the SciFact data is released under a non-commercial Creative Commons licence (CC BY-NC). **Check the repository's README/LICENSE before use**; academic, non-commercial evaluation use is the intent here.

**Official split usage.** SciFact's test labels are hidden. We therefore (i) take the official **dev** set as the held-out **test** set and (ii) split the official **train** set by *connected components of cited documents* into train / validation / conformal-calibration, so that no cited paper is shared between partitions (leakage prevention, paper Sec. VIII-A.3). If the download fails, the cell prints manual instructions and stops; **no data is fabricated**.

In [ ]:
DATA_DIR = Path(CONFIG["DATA_DIR"])
REQUIRED_FILES = ["corpus.jsonl", "claims_train.jsonl", "claims_dev.jsonl"]

MANUAL_INSTRUCTIONS = (
    "Could not download SciFact automatically.\n"
    "Manual setup: open https://github.com/allenai/scifact, follow its data-download instructions, and place\n"
    f"{', '.join(REQUIRED_FILES)} inside '{CONFIG['DATA_DIR']}/'. Then re-run this cell. "
    "(In Colab: use the Files panel to upload them.) No data will be fabricated."
)

def acquire_scifact(data_dir, url):
    """Download and extract the SciFact release unless the required files already exist."""
    data_dir.mkdir(parents=True, exist_ok=True)
    if all((data_dir / f).exists() for f in REQUIRED_FILES):
        print("SciFact files already present in", data_dir.resolve())
        return
    archive = data_dir / "scifact_data.tar.gz"
    print("Downloading SciFact from", url)
    try:
        urllib.request.urlretrieve(url, archive)
    except Exception as exc:
        raise RuntimeError(MANUAL_INSTRUCTIONS) from exc
    wanted = set(REQUIRED_FILES) | {"claims_test.jsonl"}
    with tarfile.open(archive) as tar:
        for member in tar.getmembers():
            base = os.path.basename(member.name)          # basename only -> no path traversal
            if member.isfile() and base in wanted:
                with tar.extractfile(member) as src, open(data_dir / base, "wb") as dst:
                    dst.write(src.read())
    missing = [f for f in REQUIRED_FILES if not (data_dir / f).exists()]
    if missing:
        raise RuntimeError(f"Archive did not contain {missing}. " + MANUAL_INSTRUCTIONS)

def read_jsonl(path):
    """Read a JSON-lines file with explicit error messages."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. " + MANUAL_INSTRUCTIONS)
    rows = []
    with open(path, encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"{path}:{line_no} is not valid JSON: {exc}") from exc
    return rows

acquire_scifact(DATA_DIR, CONFIG["DATA_URL"])

In [ ]:
def normalize_text(text):
    """Unicode NFKC normalisation + whitespace collapsing. Case, digits, symbols and punctuation are preserved."""
    if text is None:
        return ""
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", str(text))).strip()

# ---------------------------------- corpus (cited sources) ----------------------------------
corpus_rows = read_jsonl(DATA_DIR / "corpus.jsonl")
CORPUS = {}
for d in corpus_rows:
    if "doc_id" not in d or "abstract" not in d:
        raise ValueError("Unexpected corpus record format: " + str(list(d.keys())))
    did = int(d["doc_id"])
    if did in CORPUS:
        raise ValueError(f"Duplicate doc_id {did} in corpus")
    CORPUS[did] = {"title": normalize_text(d.get("title", "")),
                   "sents": [normalize_text(s) for s in d["abstract"]]}

# Sentence-level index: row r  <->  (doc_id, sentence_index, text)
DOC_IDS = sorted(CORPUS)
DOC_POS = {did: i for i, did in enumerate(DOC_IDS)}
SENT_ROWS, DOC_ROWS = [], {}
for did in DOC_IDS:
    for j, s in enumerate(CORPUS[did]["sents"]):
        DOC_ROWS.setdefault(did, []).append(len(SENT_ROWS))
        SENT_ROWS.append((did, j, s))
ROW_OF = {(d, j): r for r, (d, j, _) in enumerate(SENT_ROWS)}
SENT_TEXTS = [t for _, _, t in SENT_ROWS]
DOC_TEXTS = [(CORPUS[d]["title"] + ". " + " ".join(CORPUS[d]["sents"])).strip() for d in DOC_IDS]
print(f"Corpus: {len(CORPUS):,} abstracts, {len(SENT_ROWS):,} sentences")

In [ ]:
# ----------------------------------------- claims -----------------------------------------
LABELS = ["SUPPORTED", "REFUTED", "NOT_ENOUGH_INFO"]        # class ids 0, 1, 2 (used everywhere)
L2I = {l: i for i, l in enumerate(LABELS)}
SCIFACT_TO_LABEL = {"SUPPORT": "SUPPORTED", "CONTRADICT": "REFUTED"}   # original labels preserved, only renamed

def parse_claims(rows, split_name):
    """Convert raw SciFact claim records into flat dictionaries (labels preserved)."""
    out = []
    for r in rows:
        cid = int(r["id"]); claim = normalize_text(r["claim"])
        if not claim:
            raise ValueError(f"Empty claim text for id {cid}")
        labels, gold, gold_sets = [], {}, []
        for doc, sets in (r.get("evidence") or {}).items():
            for s in sets:
                if s["label"] not in SCIFACT_TO_LABEL:
                    raise ValueError(f"Unknown evidence label {s['label']!r} for claim {cid}")
                labels.append(SCIFACT_TO_LABEL[s["label"]])
                idxs = [int(i) for i in s["sentences"]]
                gold.setdefault(int(doc), set()).update(idxs)
                gold_sets.append((int(doc), tuple(sorted(idxs))))
        if labels:
            counts = Counter(labels); label = counts.most_common(1)[0][0]; conflict = len(counts) > 1
        else:
            label, conflict = "NOT_ENOUGH_INFO", False
        out.append({"claim_id": cid, "claim": claim, "label": label, "y": L2I[label],
                    "cited_doc_ids": [int(x) for x in r.get("cited_doc_ids", [])],
                    "gold_sents": {d: sorted(v) for d, v in gold.items()}, "gold_sets": gold_sets,
                    "split": split_name, "label_conflict": conflict})
    return out

def assign_groups(records):
    """Group claims by connected components of the 'claims share a cited/evidence document' graph (union-find)."""
    parent = {}
    def find(x):
        while parent.setdefault(x, x) != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def docs_of(r):
        return list(dict.fromkeys(r["cited_doc_ids"] + list(r["gold_sents"])))
    for r in records:
        docs = docs_of(r)
        for d in docs:
            find(d)
        for d in docs[1:]:
            ra, rb = find(docs[0]), find(d)
            if ra != rb:
                parent[rb] = ra
    for r in records:
        docs = docs_of(r)
        r["group"] = f"cc{find(docs[0])}" if docs else f"claim{r['claim_id']}"

pool = parse_claims(read_jsonl(DATA_DIR / "claims_train.jsonl"), "train_pool")
test_recs = parse_claims(read_jsonl(DATA_DIR / "claims_dev.jsonl"), "test")
print(f"Official train pool: {len(pool)} claims | official dev (our held-out test): {len(test_recs)} claims")

# ---- data-integrity checks (reported, not silently ignored) ----
all_claims = pool + test_recs
checks = {
    "claims citing a doc_id missing from the corpus": sum(any(d not in CORPUS for d in r["cited_doc_ids"]) for r in all_claims),
    "claims with no cited document": sum(len(r["cited_doc_ids"]) == 0 for r in all_claims),
    "claims whose gold evidence doc is not in cited_doc_ids": sum(any(d not in r["cited_doc_ids"] for d in r["gold_sents"]) for r in all_claims),
    "claims with an out-of-range gold sentence index": sum(any((d, j) not in ROW_OF for d, js in r["gold_sents"].items() for j in js) for r in all_claims),
    "claims with conflicting evidence labels": sum(r["label_conflict"] for r in all_claims),
    "duplicate claim ids across pool+test": len(all_claims) - len({r["claim_id"] for r in all_claims}),
}
for k, v in checks.items():
    print(f"  {k}: {v}")

# ---- leakage control 1: remove pool claims that cite documents also cited by test claims ----
test_docs = {d for r in test_recs for d in (r["cited_doc_ids"] + list(r["gold_sents"]))}
overlap = [r for r in pool if any(d in test_docs for d in (r["cited_doc_ids"] + list(r["gold_sents"])))]
print(f"Pool claims sharing a cited document with the test set: {len(overlap)}")
if CONFIG["DROP_TEST_DOC_OVERLAP"] and overlap:
    drop_ids = {r["claim_id"] for r in overlap}
    pool = [r for r in pool if r["claim_id"] not in drop_ids]
    print(f"  -> dropped {len(drop_ids)} claims from the training pool; {len(pool)} remain")

# ---- leakage control 2: group-wise split of the pool ----
assign_groups(pool); assign_groups(test_recs)
pool_df = pd.DataFrame(pool)
gss1 = GroupShuffleSplit(n_splits=1, test_size=1 - CONFIG["TRAIN_FRACTION"], random_state=CONFIG["SEED"])
tr_idx, rest_idx = next(gss1.split(pool_df, groups=pool_df["group"]))
rest_df = pool_df.iloc[rest_idx].reset_index(drop=True)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=CONFIG["SEED"])
va_idx, ca_idx = next(gss2.split(rest_df, groups=rest_df["group"]))

SPLITS = {"train": pool_df.iloc[tr_idx].assign(split="train").reset_index(drop=True),
          "val": rest_df.iloc[va_idx].assign(split="val").reset_index(drop=True),
          "cal": rest_df.iloc[ca_idx].assign(split="cal").reset_index(drop=True),
          "test": pd.DataFrame(test_recs)}

# claim pool = sentences of the cited abstract(s): the "known cited source" setting (paper Stage 5 onward)
def claim_pool_rows(rec):
    return [r for d in rec["cited_doc_ids"] if d in DOC_ROWS for r in DOC_ROWS[d]]

for df in SPLITS.values():
    df["pool_rows"] = [claim_pool_rows(r) for r in df.to_dict("records")]
RECS = {s: df.to_dict("records") for s, df in SPLITS.items()}

# ---- verify partition disjointness (no cited document shared between partitions) ----
docs_by_split = {s: {d for r in RECS[s] for d in (r["cited_doc_ids"] + list(r["gold_sents"]))} for s in RECS}
for a in RECS:
    for b in RECS:
        if a < b:
            shared = docs_by_split[a] & docs_by_split[b]
            if shared:
                raise RuntimeError(f"Leakage: {len(shared)} cited documents shared between {a} and {b}")
print("Leakage check passed: no cited document is shared between train / val / cal / test.")
print({s: len(v) for s, v in RECS.items()})

## 5. Data Exploration and Preprocessing

We inspect label distributions, claim lengths, evidence annotations and corpus structure. **Preprocessing** is deliberately minimal (Unicode NFKC + whitespace normalisation, done above) so that numbers, units, negations, modals and quantifiers are preserved. Original SciFact labels are only *renamed* (SUPPORT→SUPPORTED, CONTRADICT→REFUTED, no evidence→NOT_ENOUGH_INFO).

In [ ]:
def split_summary(recs_by_split):
    rows = []
    for s, recs in recs_by_split.items():
        df = pd.DataFrame(recs); counts = df["label"].value_counts()
        rows.append({"split": s, "claims": len(df), "cited-doc groups": df["group"].nunique(),
                     **{l: int(counts.get(l, 0)) for l in LABELS},
                     "mean claim words": round(df["claim"].str.split().str.len().mean(), 1),
                     "mean cited docs/claim": round(df["cited_doc_ids"].apply(len).mean(), 2)})
    return pd.DataFrame(rows)

summary_df = split_summary(RECS)
display(summary_df)

long = []
for s, recs in RECS.items():
    c = Counter(r["label"] for r in recs)
    for l in LABELS:
        long.append({"split": s, "label": l, "percent": 100 * c.get(l, 0) / max(1, len(recs))})
plt.figure(figsize=(7, 4))
sns.barplot(data=pd.DataFrame(long), x="split", y="percent", hue="label")
plt.title("Class distribution per partition (percent of claims)")
plt.xlabel("Partition"); plt.ylabel("Percent of claims"); plt.legend(title="Label")
savefig("fig_class_distribution.png")

# corpus and evidence structure
sents_per_doc = np.array([len(CORPUS[d]["sents"]) for d in DOC_IDS])
print(f"Abstracts: {len(DOC_IDS):,} | sentences/abstract: mean {sents_per_doc.mean():.1f}, median {np.median(sents_per_doc):.0f}, max {sents_per_doc.max()}")
labelled = [r for rs in RECS.values() for r in rs if r["label"] != "NOT_ENOUGH_INFO"]
print(f"Labelled claims across partitions: {len(labelled)} | gold rationale sentences per labelled claim: "
      f"mean {np.mean([sum(len(v) for v in r['gold_sents'].values()) for r in labelled]):.2f}")
has_digit = [bool(re.search(r"\d", r["claim"])) for rs in RECS.values() for r in rs]
print(f"Claims containing a digit: {np.mean(has_digit):.1%}")

## 6. Scientific Claim Processing

### 6.1 Tokenisation that preserves scientific content **[IMPL]**
Retrieval and the TF-IDF classifier use a tokenizer that (i) lower-cases, (ii) keeps numbers such as `4.5%` and hyphenated terms such as `il-6` as single tokens, and (iii) removes English stop-words **except** negations, modals, quantifiers and comparatives (`not, no, never, without, may, might, could, only, all, most, more, less, than ...`), because those carry the qualifiers that the paper's taxonomy (T1, T3, T4, T7) depends on.

### 6.2 Atomic claim decomposition: a simple, transparent demonstration **[PAPER Stages 2-4 → simplified IMPL]**
The paper decomposes multi-clause citing sentences into independently verifiable atomic claims and binds each to its governing citation anchor (Eqs. 5-6). A faithful implementation needs coreference resolution and an instruction-tuned LLM. Here we implement only a **rule-based splitter** (semicolons, `while/whereas/although/though`, `, but`, `, and`) that splits only if each part has ≥ 4 words, and **never splits on conditional markers** (`if, when, unless, because`) so that condition modifiers stay attached (paper Stage 3). Pronouns are **not** resolved, and quality is **not** evaluated (SciFact has no gold atomic claims). Decomposition is therefore *not* used in any reported metric; it is a demonstration plus descriptive statistics.

In [ ]:
TOKEN_RE = re.compile(r"[a-z0-9]+(?:[.\-/][a-z0-9]+)*%?")
# Words that sklearn's list would delete but that carry negation / modality / quantity / comparison.
KEEP_WORDS = {"not", "no", "nor", "never", "cannot", "without", "none", "nothing", "neither",
              "may", "might", "could", "can", "only", "all", "most", "some", "few", "several", "any",
              "every", "each", "more", "less", "fewer", "least", "much", "many", "than", "against", "between"}
STOPWORDS = set(ENGLISH_STOP_WORDS) - KEEP_WORDS

def tokenize(text):
    """Lower-cased tokens; numbers (4.5%), hyphenated terms (il-6) and qualifiers are preserved."""
    return [t for t in TOKEN_RE.findall(normalize_text(text).lower()) if t not in STOPWORDS]

print(tokenize("Treatment did NOT reduce IL-6 levels by 4.5% in all patients (p>0.05)."))

# ---------------- rule-based decomposition + anchor attribution (illustrative, not evaluated) ----------------
_LEAD_SUBORD = re.compile(r"^(?:while|whereas|although|though)\s+(.+?),\s+(.+)$", re.I)
_SPLIT_RE = re.compile(r";\s+|,?\s+(?:while|whereas|although|though)\s+|,\s+but\s+|\s+but\s+|,\s+and\s+", re.I)
ANCHOR_RE = re.compile(r"\[\d+(?:\s*[,\u2013\-]\s*\d+)*\]")

def decompose_claim(text, min_words=4):
    """Split a compound claim into candidate atomic parts (see markdown above for limitations)."""
    text = normalize_text(text).rstrip(".")
    m = _LEAD_SUBORD.match(text)
    if m and all(len(p.split()) >= min_words for p in m.groups()):
        parts = [m.group(1), m.group(2)]
    else:
        parts, start = [], 0
        for m in _SPLIT_RE.finditer(text):
            left, right = text[start:m.start()], text[m.end():]
            if len(left.split()) >= min_words and len(right.split()) >= min_words:
                parts.append(left); start = m.end()
        parts.append(text[start:])
    parts = [p.strip(" ,;") for p in parts if p.strip(" ,;")]
    return [p[0].upper() + p[1:] for p in parts] or [text]

def attribute_anchors(sentence):
    """Toy version of Stage 4: decompose a sentence and attach to each part the citation anchors found inside it."""
    return [{"atomic_claim": ANCHOR_RE.sub("", p).strip(" ,"), "anchors": ANCHOR_RE.findall(p)}
            for p in decompose_claim(sentence)]

# (a) REAL benchmark claims: how often does the splitter fire? (descriptive only)
train_claims = [r["claim"] for r in RECS["train"]]
n_parts = np.array([len(decompose_claim(c)) for c in train_claims])
print(f"[OBS] Train claims split into >=2 parts by the heuristic: {np.mean(n_parts >= 2):.1%} (n={len(n_parts)})")
print("[OBS] First 5 train claims that were split (not hand-picked; file order):")
for c in [c for c, n in zip(train_claims, n_parts) if n >= 2][:5]:
    print("  -", c); [print("      ->", p) for p in decompose_claim(c)]

# (b) ILLUSTRATIVE example taken from the paper's Fig. 2 (constructed text, not benchmark data)
fig2 = ("While Switch-MoE cuts pre-training time by 4x [12], it suffers from severe overfitting "
        "on fine-tuning tasks compared to dense T5 [18].")
print("\n[ILLUS] Paper Fig. 2 sentence, rule-based result:")
for row in attribute_anchors(fig2):
    print("  ", row)
print("  Note: the pronoun 'it' stays unresolved; the paper's target output has 'Switch-MoE' substituted (needs coreference).")

## 7. Evidence Retrieval Methods

Two retrieval settings, both using the **same** index and tokenizer so the comparison is controlled:

1. **Sentence retrieval inside the cited abstract(s)** (primary). The candidate pool for a claim is the sentences of its `cited_doc_ids`. This mirrors the paper's setting in which the cited source is known (Stage 5 → 7). Ground truth: gold rationale sentences (only claims with SUPPORT/CONTRADICT evidence have any).
2. **Abstract retrieval over the whole corpus** (secondary, open-domain). Ground truth: gold evidence abstracts.

**Methods.** (i) **TF-IDF + cosine** (sub-linear tf, L2-normalised). (ii) **BM25** (`rank_bm25.BM25Okapi`, k1=1.5, b=0.75). (iii) **Dense** bi-encoder cosine (optional; sentence-transformers). Sparse statistics (IDF, average length) are computed over the whole corpus *without labels*, so they leak no supervision. All retrievers are used off-the-shelf; none is trained.

**Metrics.** *Recall@K (hit-rate)*: fraction of claims for which at least one gold item is in the top K. *Recall@K (coverage)*: mean fraction of a claim's gold items found in the top K (identical to hit-rate when there is one gold item). *MRR*: mean of 1/rank of the first gold item. Both are computed on claims that have gold evidence; NEI claims have no ground truth and are excluded here (they are used in verification).

In [ ]:
class BaseRetriever:
    name = "base"
    def score(self, query, idx=None):
        """Scores of `query` against all rows (idx=None) or against the row ids in `idx`."""
        raise NotImplementedError

class TfidfRetriever(BaseRetriever):
    name = "TF-IDF"
    def __init__(self, texts):
        self.vec = TfidfVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None, sublinear_tf=True)
        self.X = self.vec.fit_transform(texts).tocsr()          # rows are L2-normalised -> dot product = cosine
    def score(self, query, idx=None):
        q = self.vec.transform([query])
        X = self.X if idx is None else self.X[list(idx)]
        return (X @ q.T).toarray().ravel()

class BM25Retriever(BaseRetriever):
    name = "BM25"
    def __init__(self, texts, k1=1.5, b=0.75):
        from rank_bm25 import BM25Okapi
        self.n = len(texts)
        self.bm25 = BM25Okapi([tokenize(t) for t in texts], k1=k1, b=b)
    def score(self, query, idx=None):
        q = tokenize(query)
        n_out = self.n if idx is None else len(idx)
        if not q:
            return np.zeros(n_out)
        if idx is None:
            return np.asarray(self.bm25.get_scores(q), dtype=float)
        ids = [int(i) for i in idx]
        try:
            return np.asarray(self.bm25.get_batch_scores(q, ids), dtype=float)
        except AttributeError:                                   # older rank_bm25 without get_batch_scores
            return np.asarray(self.bm25.get_scores(q), dtype=float)[ids]

class DenseRetriever(BaseRetriever):
    name = "Dense"
    def __init__(self, texts, model, batch_size=64):
        self.texts, self.model, self.bs = texts, model, batch_size
        self.cache, self._full = {}, None                       # embeddings are computed lazily and cached
    def _encode_rows(self, rows):
        missing = [r for r in rows if r not in self.cache]
        if missing:
            emb = self.model.encode([self.texts[r] for r in missing], batch_size=self.bs, convert_to_numpy=True,
                                    normalize_embeddings=True, show_progress_bar=len(missing) > 2000)
            for r, e in zip(missing, emb):
                self.cache[r] = e
    def score(self, query, idx=None):
        q = self.model.encode([query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)[0]
        if idx is None:
            if self._full is None:
                rows = list(range(len(self.texts))); self._encode_rows(rows)
                self._full = np.stack([self.cache[r] for r in rows])
            return self._full @ q
        rows = [int(i) for i in idx]; self._encode_rows(rows)
        return np.stack([self.cache[r] for r in rows]) @ q

# ------------------------------------------- build indexes -------------------------------------------
t0 = pd.Timestamp.now()
SENT_RET = {"TF-IDF": TfidfRetriever(SENT_TEXTS), "BM25": BM25Retriever(SENT_TEXTS)}
DOC_RET = {"TF-IDF": TfidfRetriever(DOC_TEXTS), "BM25": BM25Retriever(DOC_TEXTS)}
DENSE_AVAILABLE = False
if CONFIG["USE_DENSE"]:
    try:
        from sentence_transformers import SentenceTransformer
        _st_model = SentenceTransformer(CONFIG["DENSE_MODEL"])
        SENT_RET["Dense"] = DenseRetriever(SENT_TEXTS, _st_model)
        DOC_RET["Dense"] = DenseRetriever(DOC_TEXTS, _st_model)
        DENSE_AVAILABLE = True
    except Exception as exc:
        print(f"WARNING: dense retriever disabled ({type(exc).__name__}: {exc})")
print("Retrievers available:", list(SENT_RET), f"| index time {(pd.Timestamp.now() - t0).total_seconds():.1f}s")

# ---------------- ranked sentence lists inside each claim's candidate pool (cached) ----------------
RANK_CACHE = {}
def ranked_rows(rec, ret_name):
    """Candidate sentence rows of the claim's cited abstract(s), best first: [(row_id, score), ...]."""
    key = (rec["claim_id"], ret_name)
    if key not in RANK_CACHE:
        pool_rows = rec["pool_rows"]
        if not pool_rows:
            RANK_CACHE[key] = []
        else:
            sc = SENT_RET[ret_name].score(rec["claim"], pool_rows)
            order = np.argsort(-sc, kind="stable")                # stable: ties keep document order
            RANK_CACHE[key] = [(pool_rows[i], float(sc[i])) for i in order]
    return RANK_CACHE[key]

In [ ]:
def eval_sentence_retrieval(recs, ret_name, ks):
    """Per-claim sentence-retrieval metrics inside the cited abstract(s). Claims without gold evidence are skipped."""
    rows, skipped = [], 0
    for rec in recs:
        pool_set = set(rec["pool_rows"])
        gold = {ROW_OF[(d, j)] for d, js in rec["gold_sents"].items() for j in js if ROW_OF.get((d, j)) in pool_set}
        if not gold:
            skipped += 1; continue
        pos = {r: i + 1 for i, (r, _) in enumerate(ranked_rows(rec, ret_name))}
        ranks = sorted(pos[g] for g in gold)
        row = {"claim_id": rec["claim_id"], "group": rec["group"], "n_gold": len(gold), "rr": 1.0 / ranks[0]}
        for k in ks:
            row[f"hit@{k}"] = float(ranks[0] <= k)
            row[f"recall@{k}"] = sum(r <= k for r in ranks) / len(ranks)
        rows.append(row)
    return pd.DataFrame(rows), skipped

def eval_doc_retrieval(recs, ret, ks):
    """Per-claim abstract-level retrieval over the entire corpus. Claims without gold evidence are skipped."""
    rows = []
    for rec in recs:
        gold_pos = [DOC_POS[d] for d in rec["gold_sents"] if d in DOC_POS]
        if not gold_pos:
            continue
        order = np.argsort(-ret.score(rec["claim"]), kind="stable")
        rank_of = np.empty(len(order), dtype=int); rank_of[order] = np.arange(1, len(order) + 1)
        ranks = sorted(int(rank_of[p]) for p in gold_pos)
        row = {"claim_id": rec["claim_id"], "group": rec["group"], "n_gold": len(ranks), "rr": 1.0 / ranks[0]}
        for k in ks:
            row[f"hit@{k}"] = float(ranks[0] <= k)
            row[f"recall@{k}"] = sum(r <= k for r in ranks) / len(ranks)
        rows.append(row)
    return pd.DataFrame(rows)

def summarize_retrieval(per_claim, metric_cols):
    """Mean and cluster-bootstrap CI for each metric column."""
    out = {}
    g = per_claim["group"].values
    for col in metric_cols:
        v = per_claim[col].values
        p, lo, hi = cluster_bootstrap(lambda idx, v=v: v[idx].mean(), g)
        out[col] = (p, lo, hi)
    return out

RET_PER_CLAIM = {}        # (level, split, method) -> per-claim DataFrame
ret_rows = []
for split in ["val", "test"]:
    for name in SENT_RET:
        df, skipped = eval_sentence_retrieval(RECS[split], name, CONFIG["K_SENT"])
        if len(df) == 0:
            raise RuntimeError(f"No '{split}' claims with gold evidence inside their cited abstracts.")
        RET_PER_CLAIM[("sentence", split, name)] = df
        cols = ["rr"] + [f"{m}@{k}" for k in CONFIG["K_SENT"] for m in ("hit", "recall")]
        for col, (p, lo, hi) in summarize_retrieval(df, cols).items():
            ret_rows.append({"level": "sentence", "split": split, "method": name, "metric": col.replace("rr", "MRR"),
                             "value": p, "ci_low": lo, "ci_high": hi, "n_claims": len(df)})
t0 = pd.Timestamp.now()
for name in DOC_RET:
    df = eval_doc_retrieval(RECS["test"], DOC_RET[name], CONFIG["K_DOC"])
    if len(df) == 0:
        raise RuntimeError("No test claims with gold evidence found; check the dataset files.")
    RET_PER_CLAIM[("abstract", "test", name)] = df
    cols = ["rr"] + [f"{m}@{k}" for k in CONFIG["K_DOC"] for m in ("hit", "recall")]
    for col, (p, lo, hi) in summarize_retrieval(df, cols).items():
        ret_rows.append({"level": "abstract", "split": "test", "method": name, "metric": col.replace("rr", "MRR"),
                         "value": p, "ci_low": lo, "ci_high": hi, "n_claims": len(df)})
print(f"abstract-level retrieval over {len(DOC_IDS):,} abstracts took {(pd.Timestamp.now() - t0).total_seconds():.1f}s")

RET_RESULTS = pd.DataFrame(ret_rows)
for level, split in [("sentence", "test"), ("abstract", "test")]:
    sub = RET_RESULTS[(RET_RESULTS.level == level) & (RET_RESULTS.split == split)]
    tab = sub.assign(txt=sub.apply(lambda r: f"{r.value:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]", axis=1)) \
             .pivot(index="metric", columns="method", values="txt")
    n = int(sub.n_claims.iloc[0])
    print(f"\n[OBS] {level}-level retrieval, {split} split, n={n} claims with gold evidence (value [95% cluster-bootstrap CI])")
    display(tab)

In [ ]:
# ------------------- display top retrieved evidence for a few (randomly sampled, not curated) test claims -------------------
def show_top_evidence(rec, k=3):
    print("=" * 110)
    print(f"Claim {rec['claim_id']} [{rec['label']}]: {rec['claim']}")
    for name in SENT_RET:
        print(f"  [{name}]")
        for r, s in ranked_rows(rec, name)[:k]:
            d, j, t = SENT_ROWS[r]
            is_gold = d in rec["gold_sents"] and j in rec["gold_sents"][d]
            print(f"    {s:8.3f}  doc={d} sent={j} {'<-- GOLD RATIONALE' if is_gold else ''}\n              {t[:170]}")

with_gold = [r for r in RECS["test"] if r["gold_sents"] and r["pool_rows"]]
sample_rng = np.random.default_rng(CONFIG["SEED"])
for i in sample_rng.choice(len(with_gold), size=min(3, len(with_gold)), replace=False):
    show_top_evidence(with_gold[int(i)])

# ------------------------------------ retrieval performance figures ------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (level, ks) in zip(axes, [("sentence", CONFIG["K_SENT"]), ("abstract", CONFIG["K_DOC"])]):
    sub = RET_RESULTS[(RET_RESULTS.level == level) & (RET_RESULTS.split == "test")]
    for name in SENT_RET:
        vals = [sub[(sub.method == name) & (sub.metric == f"hit@{k}")]["value"].iloc[0] for k in ks]
        ax.plot(ks, vals, marker="o", label=name)
    ax.set_title(f"{level.capitalize()}-level retrieval: Recall@K (hit-rate), test set")
    ax.set_xlabel("K (number of retrieved items)"); ax.set_ylabel("Recall@K (hit-rate)"); ax.set_ylim(0, 1.02)
    ax.set_xticks(ks); ax.legend(title="Retriever")
savefig("fig_retrieval_recall_at_k.png")

fig, ax = plt.subplots(figsize=(7, 4))
sub = RET_RESULTS[(RET_RESULTS.metric == "MRR") & (RET_RESULTS.split == "test")].reset_index(drop=True)
sns.barplot(data=sub, x="level", y="value", hue="method", order=["sentence", "abstract"], hue_order=list(SENT_RET), ax=ax, errorbar=None)
xs = {lvl: i for i, lvl in enumerate(["sentence", "abstract"])}
n_methods = sub.method.nunique(); width = 0.8 / n_methods
for _, r in sub.iterrows():
    mi = list(SENT_RET).index(r.method)
    x = xs[r.level] - 0.4 + width * (mi + 0.5)
    ax.errorbar(x, r.value, yerr=[[r.value - r.ci_low], [r.ci_high - r.value]], color="black", capsize=3)
ax.set_title("Mean Reciprocal Rank by retriever (test set, 95% cluster-bootstrap CI)")
ax.set_xlabel("Retrieval level"); ax.set_ylabel("MRR"); ax.legend(title="Retriever")
savefig("fig_retrieval_mrr.png")

## 8. Claim-Evidence Verification Models

All verifiers answer: given a claim and evidence retrieved from its cited abstract, predict **SUPPORTED / REFUTED / NOT_ENOUGH_INFO**.

| System | Description | Provenance |
|---|---|---|
| **Majority class** | Always predicts the most frequent *training* label | **[IMPL]** reference point |
| **Retrieval-only similarity** | Predict SUPPORTED if the best retrieved sentence's TF-IDF cosine with the claim is ≥ τ, else NEI; τ tuned on validation. It *cannot* output REFUTED | **[IMPL]** operationalises "topical similarity = support" |
| **TF-IDF + LR** | Logistic regression on TF-IDF of claim, evidence, their element-wise product and their cosine. Fitted on the training partition only | **[IMPL]** traditional ML baseline |
| **NLI zero-shot** | Pretrained general-domain NLI cross-encoder, **premise = one retrieved sentence, hypothesis = claim**; labels mapped entailment→SUPPORTED, contradiction→REFUTED, neutral→NEI. The claim-level prediction uses the *most decisive* retrieved sentence (the one with the lowest neutral probability) | **[MODEL]** + **[IMPL]** aggregation |
| **NLI + synthesis** | The same NLI evidence plus relevance/confidence gates and abstention (Section 9) | **[IMPL]** |
| **NLI fine-tuned** *(optional)* | The same NLI model fine-tuned on sentence-level SciFact training pairs | **[MODEL]** + **[IMPL]**, off by default |

**Semantic similarity ≠ logical entailment.** A retrieved sentence can share most of the claim's vocabulary and still say the opposite, or only mention the topic ("effect on mortality has not been established"). Similarity measures *aboutness*; entailment asks whether the evidence *licenses* the claim. This is the paper's "topical overlap trap" (Stage 8 justification) and its Layer-3 vs Layer-4 distinction (Table I). The retrieval-only system makes the failure mode measurable, and Section 8.3 shows it on a constructed pair (**[ILLUS]**).

No scientific-domain NLI checkpoint is used by default because suitable public checkpoints and their label conventions vary; fine-tuning the general NLI model on SciFact (optional) is the domain-adaptation route provided here.

In [ ]:
# ------------------------------- NLI model wrapper (zero-shot and optional fine-tuned) -------------------------------
NLI_MODELS = {}                                   # key -> bundle ('zero-shot', 'fine-tuned')
NLI_CACHES = {"zero-shot": {}, "fine-tuned": {}}  # key -> {(claim_id, sentence_row): logits[ENT, CON, NEU]}

def load_nli(model_name):
    """Load a Hugging Face NLI cross-encoder and work out which output index is entailment / contradiction / neutral."""
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device).eval()
    id2label = {int(k): str(v).lower() for k, v in model.config.id2label.items()}
    perm = []
    for key in ("entail", "contradict", "neutral"):              # our class order: ENT, CON, NEU
        hits = [i for i, l in id2label.items() if key in l]
        if len(hits) != 1:
            raise ValueError(f"Cannot map NLI labels {id2label} to entailment/contradiction/neutral")
        perm.append(hits[0])
    return {"name": model_name, "tok": tok, "model": model, "device": device, "perm": perm}

def nli_forward(pairs, key="zero-shot"):
    """pairs: list of (premise, hypothesis). Returns logits reordered to [ENT, CON, NEU] == [SUPPORTED, REFUTED, NEI]."""
    b = NLI_MODELS[key]; bs = CONFIG["NLI_BATCH_SIZE"]; out = []
    with torch.no_grad():
        for i in range(0, len(pairs), bs):
            chunk = pairs[i:i + bs]
            enc = b["tok"]([p for p, _ in chunk], [h for _, h in chunk], truncation=True, padding=True,
                           max_length=CONFIG["NLI_MAX_LEN"], return_tensors="pt").to(b["device"])
            out.append(b["model"](**enc).logits.float().cpu().numpy()[:, b["perm"]])
    return np.vstack(out) if out else np.zeros((0, 3))

def nli_for_rows(rec, rows, key="zero-shot"):
    """NLI logits (len(rows), 3) for (premise=sentence row, hypothesis=claim), computed once and cached."""
    cache = NLI_CACHES[key]
    need = [r for r in rows if (rec["claim_id"], r) not in cache]
    if need:
        lg = nli_forward([(SENT_ROWS[r][2], rec["claim"]) for r in need], key)
        for r, l in zip(need, lg):
            cache[(rec["claim_id"], r)] = l
    return np.stack([cache[(rec["claim_id"], r)] for r in rows])

def precompute_nli(records, key="zero-shot", extra_gold=True):
    """Batch-score every sentence that any retriever can place in its top-K (and the gold sentences) for these claims."""
    needed, seen = [], set()
    for rec in records:
        rows = set()
        for name in SENT_RET:
            rows.update(r for r, _ in ranked_rows(rec, name)[:max(CONFIG["K_SENT"])])
        if extra_gold:
            rows.update(ROW_OF[(d, j)] for d, js in rec["gold_sents"].items() for j in js if (d, j) in ROW_OF)
        for r in sorted(rows):
            k = (rec["claim_id"], r)
            if k not in NLI_CACHES[key] and k not in seen:
                seen.add(k); needed.append((k, (SENT_ROWS[r][2], rec["claim"])))
    if needed:
        lg = nli_forward([p for _, p in needed], key)
        for (k, _), l in zip(needed, lg):
            NLI_CACHES[key][k] = l
    return len(needed)

In [ ]:
NLI_AVAILABLE = False
if CONFIG["USE_NLI"] and HAS_TORCH:
    try:
        t0 = pd.Timestamp.now()
        NLI_MODELS["zero-shot"] = load_nli(CONFIG["NLI_MODEL"])
        NLI_AVAILABLE = True
        print(f"Loaded NLI model {CONFIG['NLI_MODEL']} on {NLI_MODELS['zero-shot']['device']} "
              f"({(pd.Timestamp.now() - t0).total_seconds():.1f}s); output mapping -> {NLI_MODELS['zero-shot']['perm']}")
    except Exception as exc:
        print(f"WARNING: NLI model unavailable ({type(exc).__name__}: {exc}). NLI-based systems will be skipped.")
else:
    print("NLI disabled or PyTorch missing: NLI-based systems will be skipped.")

In [ ]:
class TfidfPairClassifier:
    """Traditional-ML baseline: logistic regression on TF-IDF features of (claim, evidence). Fit on training data only."""
    def __init__(self, C=1.0):
        self.C = C
    def _features(self, claims, evidences):
        Vc, Ve = self.vec.transform(claims), self.vec.transform(evidences)
        inter = Vc.multiply(Ve).tocsr()                          # shared-term features
        cos = np.asarray(inter.sum(axis=1)).reshape(-1, 1)       # vectors are L2-normalised -> cosine similarity
        return sparse.hstack([Vc, Ve, inter, sparse.csr_matrix(cos)]).tocsr()
    def fit(self, claims, evidences, y):
        self.vec = TfidfVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None, sublinear_tf=True, min_df=2)
        self.vec.fit(list(claims) + list(evidences))             # vocabulary from TRAINING text only
        self.clf = LogisticRegression(C=self.C, max_iter=2000, class_weight="balanced")
        self.clf.fit(self._features(claims, evidences), y)
        if list(self.clf.classes_) != [0, 1, 2]:
            raise ValueError(f"Training data must contain all three classes, found {self.clf.classes_}")
        return self
    def predict_proba(self, claims, evidences):
        return self.clf.predict_proba(self._features(claims, evidences))

### 8.3 Similarity versus entailment on a constructed pair **[ILLUS]**

The two pairs below are **written by hand for illustration**; they are not benchmark data and are not used in any metric. The first pair has high word overlap but the evidence does not support the claim; the second has low overlap but does.

In [ ]:
ILLUS_PAIRS = [
    ("Metformin reduces mortality in patients with type 2 diabetes.",
     "Metformin is widely prescribed to patients with type 2 diabetes, but its effect on mortality has not been established."),
    ("Metformin reduces mortality in patients with type 2 diabetes.",
     "Diabetic adults treated with the biguanide drug had lower all-cause death rates than untreated controls."),
]
tfidf_vec = SENT_RET["TF-IDF"].vec
rows = []
for claim, evid in ILLUS_PAIRS:
    cos = float((tfidf_vec.transform([claim]).multiply(tfidf_vec.transform([evid]))).sum())
    row = {"evidence (constructed)": evid[:75] + "...", "TF-IDF cosine": round(cos, 3)}
    if NLI_AVAILABLE:
        p = softmax(nli_forward([(evid, claim)])[0])
        row.update({"P(entail)": round(float(p[0]), 3), "P(contradict)": round(float(p[1]), 3), "P(neutral)": round(float(p[2]), 3)})
    rows.append(row)
display(pd.DataFrame(rows))
print("[ILLUS] Whether the model separates these pairs is an observation about this model on two hand-written examples, not a benchmark result.")

## 9. Verification Decision Synthesis

**Goal [PAPER Stages 11-12, simplified].** Combine retrieval and NLI into one decision with explicit uncertainty handling. For a claim with top-`K` retrieved sentences:

1. **Evidence relevance** `rel` = maximum TF-IDF cosine between the claim and the retrieved sentences. A retriever-independent, bounded [0,1] measure, so the same gate applies to every retriever.
2. **NLI posterior** from the *most decisive* sentence (lowest neutral probability), temperature-scaled with `T` fitted on **validation** data.
3. **Decision rule.**
   * if `rel < τ_rel` → **inconclusive** (NEI): no relevant evidence was retrieved;
   * else if the NLI argmax is SUPPORTED/REFUTED but its probability `< τ_conf` → **inconclusive** (NEI): the verdict is not confident enough to flag;
   * else → the NLI argmax.
4. `τ_rel`, `τ_conf` are chosen on the **validation** partition by macro-F1 (ties → the least aggressive thresholds) and then frozen for the test set.

This is a deliberately simple threshold rule. It is *not* the paper's composite severity score (Eq. 16), the conformal routing is evaluated separately in Section 12, and "inconclusive" is scored as the NEI class (reasons are tracked separately). With ~100-200 validation claims the tuned thresholds are noisy; this is reported, not hidden.

In [ ]:
TAU_REL_GRID = [0.0, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30]
TAU_CONF_GRID = [round(x, 2) for x in np.arange(0.0, 0.96, 0.05)]
NO_EVIDENCE_LOGITS = np.array([0.0, 0.0, 10.0])      # used only if a claim has an empty candidate pool (count is reported)

def decisive_logits(nli):
    """Pick the retrieved sentence with the lowest neutral probability; return its logits and its position."""
    if nli is None or len(nli) == 0:
        return NO_EVIDENCE_LOGITS.copy(), -1
    p = softmax(nli, axis=1)
    i = int(np.argmax(1.0 - p[:, 2]))
    return nli[i], i

def synthesize_batch(logits, rel, T, tau_rel, tau_conf):
    """Vectorised Stage-D decision rule. Returns (pred, probs, reason) with reason in {'NLI decision','low relevance','low confidence'}."""
    p = softmax(np.asarray(logits, dtype=float) / T, axis=1)
    k = p.argmax(1); pmax = p.max(1)
    low_rel = np.asarray(rel) < tau_rel
    low_conf = (~low_rel) & (k != 2) & (pmax < tau_conf)
    pred = np.where(low_rel | low_conf, 2, k)
    reason = np.where(low_rel, "low relevance", np.where(low_conf, "low confidence", "NLI decision"))
    return pred, p, reason

def tune_synthesis(logits, rel, y, T):
    """Grid search (validation data only) for (tau_rel, tau_conf) maximising macro-F1; ties -> smaller thresholds."""
    best = None
    for tr in TAU_REL_GRID:
        for tc in TAU_CONF_GRID:
            pred, _, _ = synthesize_batch(logits, rel, T, tr, tc)
            key = (macro_f1_np(y, pred), -(tr + tc))
            if best is None or key > best[0]:
                best = (key, tr, tc)
    return best[1], best[2], best[0][0]

## 10. Experimental Setup

* **Unit of evaluation:** one claim. **Evidence unit:** one abstract sentence from the claim's cited document(s). **Test set:** SciFact official dev (never used for training, tuning, calibration or model selection).
* **Training partition:** fits the TF-IDF classifier (and the optional fine-tuned NLI). **Validation:** chooses hyper-parameters (`C` of LR, similarity threshold, NLI temperature `T`, gate thresholds `τ`), and selects the *default* pipeline configuration (retriever, `K`) by the **threshold-free** zero-shot-NLI macro-F1 (or the LR macro-F1 if NLI is unavailable). **Calibration partition:** used only for split-conformal quantiles. **Test:** reported once for every configuration; selection never looks at it.
* **Conditions held constant:** the same claims, the same candidate pools, the same tokenizer and the same seed for every comparison. All retrieval/`K` configurations are reported on the test set, not only the selected one, to avoid selecting favourable results.
* **Evidence for a claim** = top-`K` sentences (`K ∈ CONFIG["K_SENT"]`) from its cited abstract(s) under the chosen retriever.

In [ ]:
RETRIEVERS = list(SENT_RET); KS = list(CONFIG["K_SENT"]); KMAX = max(KS)
BUNDLE_CACHE = {}

def build_bundle(rec, ret_name, k, key="zero-shot", with_nli=True, rows=None):
    """Everything the verifiers need for one claim: retrieved rows/text, relevance, and (optionally) NLI logits."""
    if rows is None:
        ranked = ranked_rows(rec, ret_name)[:k]
        rows = [r for r, _ in ranked]
        top_score = ranked[0][1] if ranked else 0.0
    else:
        top_score = float("nan")
    texts = [SENT_ROWS[r][2] for r in rows]
    rel = float(np.max(SENT_RET["TF-IDF"].score(rec["claim"], rows))) if rows else 0.0
    nli = nli_for_rows(rec, rows, key) if (with_nli and rows and key in NLI_MODELS) else None
    return {"claim_id": rec["claim_id"], "rows": rows, "texts": texts, "evidence": " ".join(texts),
            "rel": rel, "top_score": top_score, "nli": nli}

def get_bundles(split, ret_name, k, key="zero-shot", with_nli=True):
    ck = (split, ret_name, k, key if with_nli else None)
    if ck not in BUNDLE_CACHE:
        BUNDLE_CACHE[ck] = [build_bundle(rec, ret_name, k, key, with_nli) for rec in RECS[split]]
    return BUNDLE_CACHE[ck]

# candidate-pool sanity report
for s in RECS:
    empty = sum(len(r["pool_rows"]) == 0 for r in RECS[s])
    print(f"{s}: claims with an empty candidate pool = {empty} (such claims receive 'no evidence' -> NEI)")

if NLI_AVAILABLE:
    t0 = pd.Timestamp.now()
    for s in ["val", "cal", "test"]:
        n_new = precompute_nli(RECS[s], "zero-shot")
        print(f"NLI scored {n_new:,} (sentence, claim) pairs for '{s}'")
    print(f"NLI scoring time {(pd.Timestamp.now() - t0).total_seconds():.1f}s")

### 10.1 Optional advanced component: fine-tuning the NLI model on SciFact (off by default)

Enable with `CONFIG["RUN_FINETUNE"] = True` (GPU strongly recommended). Sentence-level training pairs are built from **training claims only**: gold rationale sentences get the claim's label (SUPPORTED/REFUTED); the other top-5 BM25-retrieved sentences of the same claim, and all retrieved sentences of NEI claims, are labelled neutral. Fixed epochs, no early stopping on test data. This is a plain PyTorch loop and has not been tuned.

In [ ]:
def build_finetune_pairs(records, ret_name="BM25", top_n=5):
    """Sentence-level (premise, hypothesis, class_id) triples from training claims only."""
    triples = []
    for rec in records:
        gold_rows = {ROW_OF[(d, j)] for d, js in rec["gold_sents"].items() for j in js if (d, j) in ROW_OF}
        rows = sorted(gold_rows | {r for r, _ in ranked_rows(rec, ret_name)[:top_n]})
        for r in rows:
            cls = rec["y"] if r in gold_rows else 2          # non-gold sentences are neutral w.r.t. the claim
            triples.append((SENT_ROWS[r][2], rec["claim"], cls))
    return triples

def finetune_nli(base_name, triples, epochs, lr, batch_size):
    """Fine-tune a fresh copy of the NLI model; returns a bundle usable by nli_forward."""
    bundle = load_nli(base_name)
    tok, model, device, perm = bundle["tok"], bundle["model"], bundle["device"], bundle["perm"]
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    gen = torch.Generator().manual_seed(CONFIG["SEED"])
    for ep in range(epochs):
        order = torch.randperm(len(triples), generator=gen).tolist(); total = 0.0
        for i in range(0, len(order), batch_size):
            batch = [triples[j] for j in order[i:i + batch_size]]
            enc = tok([b[0] for b in batch], [b[1] for b in batch], truncation=True, padding=True,
                      max_length=CONFIG["NLI_MAX_LEN"], return_tensors="pt").to(device)
            labels = torch.tensor([perm[b[2]] for b in batch], device=device)   # our class id -> this model's output index
            loss = model(**enc, labels=labels).loss
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad(); total += float(loss) * len(batch)
        print(f"  fine-tune epoch {ep + 1}/{epochs}: mean loss {total / len(order):.4f}")
    model.eval()
    return bundle

if CONFIG["RUN_FINETUNE"] and NLI_AVAILABLE:
    triples = build_finetune_pairs(RECS["train"])
    print(f"Fine-tuning on {len(triples):,} sentence-level training pairs from {len(RECS['train'])} training claims ...")
    NLI_MODELS["fine-tuned"] = finetune_nli(CONFIG["NLI_MODEL"], triples, CONFIG["FT_EPOCHS"], CONFIG["FT_LR"], CONFIG["FT_BATCH"])
    for s in ["val", "cal", "test"]:
        precompute_nli(RECS[s], "fine-tuned")
else:
    print("Fine-tuning skipped (CONFIG['RUN_FINETUNE'] is False or NLI unavailable).")

## 11. Evaluation Metrics

* **Accuracy**: share of claims with the correct label. Sensitive to class balance.
* **Precision / Recall / F1 per class and macro-average**: macro averages treat SUPPORTED, REFUTED and NEI equally, so a system that ignores the rarer class is penalised. **Macro-F1 is the primary metric.**
* **Weighted F1**: per-class F1 weighted by class frequency (closer to accuracy).
* **Confusion matrix**: rows = true class, columns = predicted class.
* **Recall@K, MRR**: retrieval quality (Section 7).
* **Calibration**: *Expected Calibration Error* (ECE, equal-width bins on the maximum-class probability), *Brier score* (mean squared error of the probability vector; lower is better) and *negative log-likelihood*. ECE is noisy with a few hundred claims; we report it with a cluster-bootstrap interval.
* **Split-conformal sets**: empirical coverage, mean set size, singleton fraction, abstention rate (`|C| ≠ 1`).
* **Uncertainty of estimates**: 95% **cluster-bootstrap** intervals (groups = connected components of cited documents).

In [ ]:
def classification_metrics(y, p):
    pm, rm, fm, _ = precision_recall_fscore_support(y, p, labels=[0, 1, 2], average="macro", zero_division=0)
    return {"accuracy": accuracy_score(y, p), "macro_precision": pm, "macro_recall": rm, "macro_f1": fm,
            "weighted_f1": f1_score(y, p, labels=[0, 1, 2], average="weighted", zero_division=0)}

def per_class_table(y, p):
    pr, rc, f1, sup = precision_recall_fscore_support(y, p, labels=[0, 1, 2], average=None, zero_division=0)
    return pd.DataFrame({"class": LABELS, "precision": pr, "recall": rc, "f1": f1, "support": sup})

def expected_calibration_error(probs, y, n_bins=None):
    n_bins = n_bins or CONFIG["ECE_BINS"]
    probs = np.asarray(probs); y = np.asarray(y)
    conf = probs.max(1); correct = (probs.argmax(1) == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1); ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)

def brier_score(probs, y):
    probs = np.asarray(probs); onehot = np.eye(probs.shape[1])[np.asarray(y)]
    return float(np.mean(np.sum((probs - onehot) ** 2, axis=1)))

def negative_log_likelihood(probs, y):
    probs = np.asarray(probs)
    return float(-np.mean(np.log(probs[np.arange(len(y)), np.asarray(y)] + 1e-12)))

def fit_temperature(logits, y):
    """Single-parameter temperature scaling fitted by minimising NLL on the given (validation) data."""
    logits = np.asarray(logits, dtype=float); y = np.asarray(y)
    f = lambda T: negative_log_likelihood(softmax(logits / T, axis=1), y)
    return float(minimize_scalar(f, bounds=(0.05, 20.0), method="bounded").x)

# --- self-check: the fast numpy macro-F1 used inside bootstraps must equal scikit-learn's ---
_rng = np.random.default_rng(0); _y = _rng.integers(0, 3, 200); _p = _rng.integers(0, 3, 200)
assert abs(macro_f1_np(_y, _p) - classification_metrics(_y, _p)["macro_f1"]) < 1e-12, "macro-F1 implementations disagree"
print("Metric self-check passed (numpy macro-F1 == sklearn macro-F1).")

## 12. Experimental Results

Everything below is computed at run time. Systems are evaluated on the **test** partition for *every* (retriever, `K`) configuration; validation numbers are stored in `results/` as well. The *selected default* configuration (by validation, see Section 10) is only used to keep figures readable.

In [ ]:
SYS_MAJ, SYS_RET, SYS_LR = "Majority class", "Retrieval-only similarity", "TF-IDF + LR"
SYS_NLI, SYS_SYN = "NLI zero-shot", "NLI + synthesis"
SYS_NLI_FT, SYS_SYN_FT = "NLI fine-tuned", "NLI fine-tuned + synthesis"
SYS_ORACLE = "NLI zero-shot (oracle evidence, diagnostic)"

SYS_ROWS, PRED_STORE, TUNED = [], {}, {}

def store_result(system, ret, K, split, recs, pred, prob=None, extra=None):
    y = np.array([r["y"] for r in recs]); pred = np.asarray(pred, dtype=int)
    SYS_ROWS.append({"system": system, "retriever": ret, "K": K, "split": split, "n": len(y), **classification_metrics(y, pred)})
    PRED_STORE[(system, ret, K, split)] = {"y": y, "pred": pred, "prob": prob, "extra": extra or {},
                                           "groups": np.array([r["group"] for r in recs]),
                                           "claim_ids": np.array([r["claim_id"] for r in recs])}

y_train = np.array([r["y"] for r in RECS["train"]])
MAJORITY = int(np.bincount(y_train, minlength=3).argmax())
for s in ("val", "test"):
    store_result(SYS_MAJ, "-", "-", s, RECS[s], [MAJORITY] * len(RECS[s]))
print("Majority class learned from TRAIN only:", LABELS[MAJORITY])

def run_nonnli_config(ret, K):
    """Retrieval-only similarity heuristic and TF-IDF+LR for one (retriever, K) configuration."""
    y_v = np.array([r["y"] for r in RECS["val"]])
    b_tr, b_v, b_te = (get_bundles(s, ret, K, with_nli=False) for s in ("train", "val", "test"))
    # --- retrieval-only: SUPPORTED if best-sentence similarity >= tau else NEI ---
    rel_v = np.array([b["rel"] for b in b_v]); rel_t = np.array([b["rel"] for b in b_te])
    taus = np.unique(np.concatenate([[0.0], np.quantile(rel_v, np.linspace(0, 1, 41))]))
    tau = max(taus, key=lambda t: (macro_f1_np(y_v, np.where(rel_v >= t, 0, 2)), -t))
    TUNED[(SYS_RET, ret, K)] = {"tau": float(tau)}
    store_result(SYS_RET, ret, K, "val", RECS["val"], np.where(rel_v >= tau, 0, 2))
    store_result(SYS_RET, ret, K, "test", RECS["test"], np.where(rel_t >= tau, 0, 2))
    # --- TF-IDF + logistic regression: C chosen on validation ---
    c_tr, c_v, c_te = ([r["claim"] for r in RECS[s]] for s in ("train", "val", "test"))
    e_tr, e_v, e_te = ([b["evidence"] for b in bs] for bs in (b_tr, b_v, b_te))
    best = None
    for C in (0.1, 1.0, 10.0):
        m = TfidfPairClassifier(C).fit(c_tr, e_tr, y_train)
        f = macro_f1_np(y_v, m.predict_proba(c_v, e_v).argmax(1))
        if best is None or f > best[0]:
            best = (f, C, m)
    TUNED[(SYS_LR, ret, K)] = {"C": best[1]}
    for s, c, e in (("val", c_v, e_v), ("test", c_te, e_te)):
        pr = best[2].predict_proba(c, e)
        store_result(SYS_LR, ret, K, s, RECS[s], pr.argmax(1), prob=pr)

def run_nli_config(ret, K, key="zero-shot"):
    """NLI argmax and NLI + synthesis for one (retriever, K) configuration."""
    names = (SYS_NLI, SYS_SYN) if key == "zero-shot" else (SYS_NLI_FT, SYS_SYN_FT)
    b_v, b_te = get_bundles("val", ret, K, key), get_bundles("test", ret, K, key)
    L_v = np.stack([decisive_logits(b["nli"])[0] for b in b_v]); L_t = np.stack([decisive_logits(b["nli"])[0] for b in b_te])
    rel_v = np.array([b["rel"] for b in b_v]); rel_t = np.array([b["rel"] for b in b_te])
    y_v = np.array([r["y"] for r in RECS["val"]])
    store_result(names[0], ret, K, "val", RECS["val"], L_v.argmax(1), prob=softmax(L_v, axis=1))
    store_result(names[0], ret, K, "test", RECS["test"], L_t.argmax(1), prob=softmax(L_t, axis=1))
    T = fit_temperature(L_v, y_v)                                  # temperature fitted on validation only
    tau_rel, tau_conf, f_val = tune_synthesis(L_v, rel_v, y_v, T)  # thresholds tuned on validation only
    TUNED[(names[1], ret, K)] = {"T": T, "tau_rel": tau_rel, "tau_conf": tau_conf, "val_macro_f1": f_val}
    for s, L, rel in (("val", L_v, rel_v), ("test", L_t, rel_t)):
        pred, p, reason = synthesize_batch(L, rel, T, tau_rel, tau_conf)
        store_result(names[1], ret, K, s, RECS[s], pred, prob=p, extra={"reason": reason})
    return int(sum(b["nli"] is None for b in b_v + b_te))

t0 = pd.Timestamp.now(); n_empty = 0
for ret in RETRIEVERS:
    for K in KS:
        run_nonnli_config(ret, K)
        if NLI_AVAILABLE:
            n_empty += run_nli_config(ret, K, "zero-shot")
            if "fine-tuned" in NLI_MODELS:
                run_nli_config(ret, K, "fine-tuned")
print(f"Grid finished in {(pd.Timestamp.now() - t0).total_seconds():.1f}s | claims with no evidence at all (val+test, summed over configs): {n_empty}")

SYS_DF = pd.DataFrame(SYS_ROWS)

# ---- default configuration chosen on VALIDATION (threshold-free system) ----
selector = SYS_NLI if NLI_AVAILABLE else SYS_LR
cand = SYS_DF[(SYS_DF.system == selector) & (SYS_DF.split == "val")].copy()
cand["K"] = cand["K"].astype(int)
best_row = cand.sort_values(["macro_f1", "K"], ascending=[False, True]).iloc[0]
DEFAULT = {"ret": best_row["retriever"], "K": int(best_row["K"])}
print(f"[selection] default configuration = retriever {DEFAULT['ret']}, K={DEFAULT['K']} "
      f"(chosen by validation macro-F1 of '{selector}' = {best_row['macro_f1']:.3f}; n_val={int(best_row['n'])})")

In [ ]:
def pred_entry(system, split, ret=None, K=None):
    if system == SYS_MAJ:
        return PRED_STORE[(system, "-", "-", split)]
    return PRED_STORE[(system, ret or DEFAULT["ret"], K if K is not None else DEFAULT["K"], split)]

def available_systems():
    names = [SYS_MAJ, SYS_RET, SYS_LR]
    if NLI_AVAILABLE:
        names += [SYS_NLI, SYS_SYN]
        if "fine-tuned" in NLI_MODELS:
            names += [SYS_NLI_FT, SYS_SYN_FT]
    return names

def fmt_ci(t, d=3):
    return f"{t[0]:.{d}f} [{t[1]:.{d}f}, {t[2]:.{d}f}]"

def main_table(split="test"):
    rows, num = [], []
    for s in available_systems():
        e = pred_entry(s, split); y, p, g = e["y"], e["pred"], e["groups"]
        acc = cluster_bootstrap(lambda idx: accuracy_np(y[idx], p[idx]), g)
        f1 = cluster_bootstrap(lambda idx: macro_f1_np(y[idx], p[idx]), g)
        m = classification_metrics(y, p)
        rows.append({"system": s, "n": len(y), "accuracy [95% CI]": fmt_ci(acc), "macro_precision": round(m["macro_precision"], 3),
                     "macro_recall": round(m["macro_recall"], 3), "macro_F1 [95% CI]": fmt_ci(f1),
                     "weighted_F1": round(m["weighted_f1"], 3)})
        num.append({"system": s, "acc": acc, "f1": f1})
    return pd.DataFrame(rows), num

print(f"[OBS] Default configuration {DEFAULT}; TEST partition. 95% CIs: cluster bootstrap over cited-document groups.")
MAIN_TABLE, MAIN_NUM = main_table("test")
display(MAIN_TABLE)

print("\n[OBS] Tuned parameters (all chosen on validation data):")
for k, v in TUNED.items():
    if k[1] == DEFAULT["ret"] and k[2] == DEFAULT["K"]:
        print("  ", k[0], {a: round(b, 4) for a, b in v.items()})

print("\n[OBS] Per-class metrics, test, default configuration:")
for s in available_systems():
    e = pred_entry(s, "test"); print(f"\n{s}"); display(per_class_table(e["y"], e["pred"]).round(3))

In [ ]:
# ------------- full grid: every retriever x K x system on the test set (no cherry-picking) -------------
grid = SYS_DF[(SYS_DF.split == "test") & (SYS_DF.system != SYS_MAJ)]
print("[OBS] Macro-F1 on the TEST set for every configuration (retriever, K):")
display(grid.pivot_table(index=["retriever", "K"], columns="system", values="macro_f1").round(3))
print("[OBS] Accuracy on the TEST set for every configuration:")
display(grid.pivot_table(index=["retriever", "K"], columns="system", values="accuracy").round(3))

# ----------------------------------- Figure: model comparison with CIs -----------------------------------
fig, ax = plt.subplots(figsize=(9, 4.5))
names = [d["system"] for d in MAIN_NUM]; vals = [d["f1"][0] for d in MAIN_NUM]
err = np.array([[d["f1"][0] - d["f1"][1] for d in MAIN_NUM], [d["f1"][2] - d["f1"][0] for d in MAIN_NUM]])
ax.bar(range(len(names)), vals, yerr=err, capsize=4, color=sns.color_palette("deep", len(names)), label="Macro-F1 (95% CI)")
ax.set_xticks(range(len(names))); ax.set_xticklabels([n.replace(" ", "\n", 1) for n in names], fontsize=8)
ax.set_ylabel("Macro-F1 (test set)"); ax.set_xlabel("Verification system")
ax.set_title(f"Verification quality, default configuration ({DEFAULT['ret']}, K={DEFAULT['K']})")
ax.legend(loc="upper left"); ax.set_ylim(0, 1)
savefig("fig_model_comparison.png")

# ----------------------------------------- Figure: K and retriever effects -----------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, sysname in zip(axes, [s for s in (SYS_NLI, SYS_LR) if s in available_systems()]):
    for ret in RETRIEVERS:
        sub = SYS_DF[(SYS_DF.system == sysname) & (SYS_DF.split == "test") & (SYS_DF.retriever == ret)].sort_values("K")
        ax.plot(sub["K"].astype(int), sub["macro_f1"], marker="o", label=ret)
    ax.set_title(f"{sysname}: macro-F1 vs evidence depth K (test)")
    ax.set_xlabel("K (retrieved sentences used as evidence)"); ax.set_ylabel("Macro-F1"); ax.set_xticks(KS); ax.legend(title="Retriever")
savefig("fig_k_and_retriever_effect.png")

In [ ]:
# ----------------------------------------- Figure: confusion matrices -----------------------------------------
show = [s for s in (SYS_RET, SYS_LR, SYS_NLI, SYS_SYN) if s in available_systems()]
fig, axes = plt.subplots(1, len(show), figsize=(4.2 * len(show), 3.8))
axes = np.atleast_1d(axes)
short = ["SUPP", "REFU", "NEI"]
for ax, s in zip(axes, show):
    e = pred_entry(s, "test"); cm = confusion_np(e["y"], e["pred"])
    norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    sns.heatmap(norm, annot=cm, fmt="d", cmap="Blues", vmin=0, vmax=1, cbar=False, ax=ax, xticklabels=short, yticklabels=short)
    ax.set_title(f"{s}\n(test, n={len(e['y'])})", fontsize=9); ax.set_xlabel("Predicted label"); ax.set_ylabel("True label")
fig.suptitle("Confusion matrices (cell text = counts, shading = row-normalised share)", y=1.03)
savefig("fig_confusion_matrices.png")

# ---------------- Retrieval-only vs retrieval + NLI (the paper's 'topical overlap trap'), explicit table ----------------
if NLI_AVAILABLE:
    rows = []
    for s in (SYS_RET, SYS_NLI, SYS_SYN):
        e = pred_entry(s, "test"); pc = per_class_table(e["y"], e["pred"]).set_index("class")
        rows.append({"system": s, "macro_F1": round(macro_f1_np(e["y"], e["pred"]), 3),
                     "F1 SUPPORTED": round(pc.loc["SUPPORTED", "f1"], 3), "F1 REFUTED": round(pc.loc["REFUTED", "f1"], 3),
                     "F1 NEI": round(pc.loc["NOT_ENOUGH_INFO", "f1"], 3), "REFUTED predictions": int((e["pred"] == 1).sum())})
    print("[OBS] Retrieval-only versus retrieval-plus-NLI (test, default configuration):")
    display(pd.DataFrame(rows))

    # ----------------------------------- Figure: confidence distribution -----------------------------------
    e = pred_entry(SYS_SYN, "test"); conf = e["prob"].max(1); ok = e["pred"] == e["y"]
    plt.figure(figsize=(7, 4))
    bins = np.linspace(0, 1, 21)
    plt.hist(conf[ok], bins=bins, alpha=0.65, label=f"Correct (n={int(ok.sum())})")
    plt.hist(conf[~ok], bins=bins, alpha=0.65, label=f"Incorrect (n={int((~ok).sum())})")
    plt.title("Temperature-scaled NLI confidence (max class probability), test set")
    plt.xlabel("Confidence of predicted class"); plt.ylabel("Number of claims"); plt.legend(title="Final decision")
    savefig("fig_confidence_distribution.png")

    # ----------------------- Stage D: how often did the gates fire? -----------------------
    reasons = pd.Series(e["extra"]["reason"]).value_counts().rename_axis("reason").reset_index(name="claims")
    print("[OBS] Source of NLI + synthesis decisions on the test set (gates fire -> inconclusive/NEI):"); display(reasons)

    # ----------------------- similarity score by gold label (descriptive) -----------------------
    bt = get_bundles("test", DEFAULT["ret"], DEFAULT["K"], with_nli=False)
    dfp = pd.DataFrame({"label": [r["label"] for r in RECS["test"]], "relevance": [b["rel"] for b in bt]})
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=dfp, x="label", y="relevance", order=LABELS)
    plt.title("Claim-to-evidence TF-IDF similarity by gold label (test set)")
    plt.xlabel("Gold label"); plt.ylabel("Max TF-IDF cosine of retrieved evidence")
    savefig("fig_similarity_by_label.png")

In [ ]:
# ---------------- Diagnostic only: NLI on GOLD rationale sentences (upper bound for the retrieval step) ----------------
# For claims WITH gold evidence the premise set is the gold rationale; NEI claims have none, so the default retriever's
# top-K is used. Because the existence of gold evidence already reveals "not NEI", this is a diagnostic ceiling for the
# effect of retrieval errors, not a deployable system.
if NLI_AVAILABLE:
    def oracle_rows(rec, ret, k):
        rows = sorted({ROW_OF[(d, j)] for d, js in rec["gold_sents"].items() for j in js if (d, j) in ROW_OF})
        return rows if rows else [r for r, _ in ranked_rows(rec, ret)[:k]]
    for s in ("val", "test"):
        bundles = [build_bundle(rec, DEFAULT["ret"], DEFAULT["K"], rows=oracle_rows(rec, DEFAULT["ret"], DEFAULT["K"])) for rec in RECS[s]]
        L = np.stack([decisive_logits(b["nli"])[0] for b in bundles])
        store_result(SYS_ORACLE, DEFAULT["ret"], DEFAULT["K"], s, RECS[s], L.argmax(1), prob=softmax(L, axis=1))
    comp = []
    for s in (SYS_NLI, SYS_ORACLE):
        e = pred_entry(s, "test"); comp.append({"evidence": "retrieved top-K" if s == SYS_NLI else "gold rationale (NEI: retrieved)",
                                                "n": len(e["y"]), "accuracy": round(accuracy_np(e["y"], e["pred"]), 3),
                                                "macro_F1": round(macro_f1_np(e["y"], e["pred"]), 3)})
    print("[OBS] Effect of evidence quality on zero-shot NLI (test):"); display(pd.DataFrame(comp))
    SYS_DF = pd.DataFrame(SYS_ROWS)

In [ ]:
# ----------------------------- Calibration (temperature scaling) and split conformal prediction -----------------------------
CAL_ROWS, CONF_ROWS = [], []
if NLI_AVAILABLE:
    ret, K = DEFAULT["ret"], DEFAULT["K"]
    def split_logits(split):
        bs = get_bundles(split, ret, K, "zero-shot")
        return (np.stack([decisive_logits(b["nli"])[0] for b in bs]), np.array([r["y"] for r in RECS[split]]),
                np.array([r["group"] for r in RECS[split]]))
    L_v, y_v, g_v = split_logits("val"); L_c, y_c, g_c = split_logits("cal"); L_t, y_t, g_t = split_logits("test")
    T = TUNED[(SYS_SYN, ret, K)]["T"]
    p_raw, p_ts = softmax(L_t, axis=1), softmax(L_t / T, axis=1)
    print(f"[OBS] Temperature fitted on validation (n={len(y_v)}): T = {T:.3f}  (T>1 softens, T<1 sharpens)")
    for name, p in (("raw softmax", p_raw), ("temperature-scaled", p_ts)):
        ece = cluster_bootstrap(lambda idx, p=p: expected_calibration_error(p[idx], y_t[idx]), g_t)
        CAL_ROWS.append({"posterior": name, "n_test": len(y_t), "ECE [95% CI]": fmt_ci(ece), "Brier": round(brier_score(p, y_t), 4),
                         "NLL": round(negative_log_likelihood(p, y_t), 4), "accuracy": round(accuracy_np(y_t, p.argmax(1)), 3)})
    CAL_TABLE = pd.DataFrame(CAL_ROWS); display(CAL_TABLE)

    # reliability diagram
    fig, ax = plt.subplots(figsize=(5.5, 5))
    edges = np.linspace(0, 1, CONFIG["ECE_BINS"] + 1)
    for name, p, mk in (("raw softmax", p_raw, "o"), ("temperature-scaled", p_ts, "s")):
        conf = p.max(1); corr = (p.argmax(1) == y_t).astype(float); xs, ys = [], []
        for lo, hi in zip(edges[:-1], edges[1:]):
            m = (conf > lo) & (conf <= hi)
            if m.sum() >= 3:
                xs.append(conf[m].mean()); ys.append(corr[m].mean())
        ax.plot(xs, ys, marker=mk, label=name)
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    ax.set_title("Reliability diagram (test set; bins with >= 3 claims)")
    ax.set_xlabel("Mean predicted confidence"); ax.set_ylabel("Empirical accuracy"); ax.legend()
    savefig("fig_reliability.png")

    # split conformal prediction: nonconformity s = 1 - P(true class); calibrated on the CAL partition only
    p_cal = softmax(L_c / T, axis=1); n_c = len(y_c)
    s_cal = 1.0 - p_cal[np.arange(n_c), y_c]
    for alpha in (0.05, CONFIG["CONFORMAL_ALPHA"], 0.20):
        level = min(1.0, math.ceil((n_c + 1) * (1 - alpha)) / n_c)
        qhat = float(np.quantile(s_cal, level, method="higher"))
        sets = (1.0 - p_ts) <= qhat                                       # boolean (n_test, 3)
        cover = sets[np.arange(len(y_t)), y_t].astype(float); size = sets.sum(1)
        cov_ci = cluster_bootstrap(lambda idx: cover[idx].mean(), g_t)
        single = size == 1
        CONF_ROWS.append({"alpha": alpha, "target coverage": 1 - alpha, "n_cal": n_c, "n_test": len(y_t), "q_hat": round(qhat, 3),
                          "empirical coverage [95% CI]": fmt_ci(cov_ci), "mean set size": round(float(size.mean()), 3),
                          "singleton share": round(float(single.mean()), 3), "empty-set share": round(float((size == 0).mean()), 3),
                          "abstain share (|C| != 1)": round(float((~single).mean()), 3),
                          "accuracy on singletons": round(float((p_ts.argmax(1)[single] == y_t[single]).mean()), 3) if single.any() else float("nan")})
    CONF_TABLE = pd.DataFrame(CONF_ROWS)
    print("[OBS] Split-conformal prediction sets (paper Eqs. 14-15) on top of the NLI posterior:"); display(CONF_TABLE)
    print("Note: the finite-sample guarantee is MARGINAL and needs exchangeability between calibration and test claims;"
          " SciFact's train/dev partitions come from different annotation batches, so empirical coverage can deviate from the target.")
else:
    CAL_TABLE = CONF_TABLE = pd.DataFrame()
    print("NLI unavailable: calibration and conformal analysis skipped.")

## 13. Statistical Analysis

* **Reporting.** Every table states `n`. Intervals are 95% percentile **cluster-bootstrap** intervals (resampling groups of claims that share cited documents; `N_BOOT` replicates, fixed seed).
* **Pre-specified comparisons (system level).** NLI vs TF-IDF+LR, NLI vs retrieval-only, NLI+synthesis vs NLI, NLI+synthesis vs retrieval-only, all at the validation-selected default configuration on the test set. Each is tested with an **exact McNemar test** on paired per-claim correctness, adjusted with **Holm-Bonferroni** across the family. In addition we report the paired cluster-bootstrap CI of the macro-F1 difference.
* **Assumptions.** McNemar requires paired outcomes and independent discordant pairs; claims citing the same abstract violate independence mildly, which is why the cluster-bootstrap CI is the primary uncertainty statement. McNemar tests **accuracy**, whereas the CI is for **macro-F1**; the two can disagree.
* **Interpretation rule.** A difference is called *statistically supported* only if the Holm-adjusted p < 0.05 **and** the macro-F1 CI excludes 0. Everything else is a *descriptive difference*.
* **Retrieval comparisons** are paired by claim; we report the cluster-bootstrap CI of the metric difference (no p-values). With one test partition and no repeated training runs, differences between *models* cannot be separated from sampling variability of the test claims beyond what these intervals show. The tuned thresholds also carry validation-set noise that is **not** captured.

In [ ]:
def compare_systems(pairs, split="test", alpha=0.05):
    rows = []
    for a, b in pairs:
        ea, eb = pred_entry(a, split), pred_entry(b, split)
        if not (ea["claim_ids"] == eb["claim_ids"]).all():
            raise RuntimeError("Systems are not evaluated on identical claims")
        y, pa, pb = ea["y"], ea["pred"], eb["pred"]
        n10, n01, p = mcnemar_exact(pa == y, pb == y)
        d = cluster_bootstrap(lambda idx: macro_f1_np(y[idx], pa[idx]) - macro_f1_np(y[idx], pb[idx]), ea["groups"])
        rows.append({"system A": a, "system B": b, "n": len(y), "acc A": round(accuracy_np(y, pa), 3), "acc B": round(accuracy_np(y, pb), 3),
                     "A right/B wrong": n10, "A wrong/B right": n01, "McNemar p (raw)": p,
                     "macro-F1 diff A-B [95% CI]": fmt_ci(d), "_lo": d[1], "_hi": d[2]})
    adj = holm_adjust([r["McNemar p (raw)"] for r in rows])
    for r, pa_ in zip(rows, adj):
        r["McNemar p (Holm)"] = pa_
        sig_p = pa_ < alpha; sig_ci = (r["_lo"] > 0) or (r["_hi"] < 0)
        r["reading"] = ("statistically supported" if (sig_p and sig_ci) else
                        "mixed evidence (descriptive)" if (sig_p or sig_ci) else "descriptive difference only")
        del r["_lo"], r["_hi"]
    return pd.DataFrame(rows)

if NLI_AVAILABLE:
    PAIRS = [(SYS_NLI, SYS_LR), (SYS_NLI, SYS_RET), (SYS_SYN, SYS_NLI), (SYS_SYN, SYS_RET)]
else:
    PAIRS = [(SYS_LR, SYS_RET), (SYS_LR, SYS_MAJ), (SYS_RET, SYS_MAJ)]
STAT_SYS = compare_systems(PAIRS)
STAT_SYS["McNemar p (raw)"] = STAT_SYS["McNemar p (raw)"].map(lambda v: f"{v:.4f}")
STAT_SYS["McNemar p (Holm)"] = STAT_SYS["McNemar p (Holm)"].map(lambda v: f"{v:.4f}")
print(f"[OBS] System comparisons, test set, default configuration {DEFAULT} (family of {len(PAIRS)} tests, Holm-adjusted):")
display(STAT_SYS)

# ---------------- paired retrieval comparisons (cluster-bootstrap CI of the metric difference) ----------------
RET_STAT_ROWS = []
def paired_retrieval(level, split, ref_cols):
    names = list(SENT_RET)
    for i, a in enumerate(names):
        for b in names[:i]:
            da, db = RET_PER_CLAIM[(level, split, a)], RET_PER_CLAIM[(level, split, b)]
            m = da.merge(db, on="claim_id", suffixes=("_a", "_b"))
            if len(m) == 0:
                continue
            for col in ref_cols:
                va, vb = m[f"{col}_a"].values, m[f"{col}_b"].values
                p, lo, hi = cluster_bootstrap(lambda idx: (va[idx] - vb[idx]).mean(), m["group_a"].values)
                RET_STAT_ROWS.append({"level": level, "metric": col.replace("rr", "MRR"), "A": a, "B": b, "n_claims": len(m),
                                      "mean diff A-B": p, "ci_low": lo, "ci_high": hi, "CI excludes 0": bool(lo > 0 or hi < 0)})
k_s = 3 if 3 in CONFIG["K_SENT"] else CONFIG["K_SENT"][len(CONFIG["K_SENT"]) // 2]
k_d = 10 if 10 in CONFIG["K_DOC"] else CONFIG["K_DOC"][len(CONFIG["K_DOC"]) // 2]
paired_retrieval("sentence", "test", ["rr", f"hit@{k_s}"])
paired_retrieval("abstract", "test", ["rr", f"hit@{k_d}"])
RET_STAT = pd.DataFrame(RET_STAT_ROWS)
print("[OBS] Paired retrieval differences, test set (mean difference A-B with 95% cluster-bootstrap CI):")
display(RET_STAT.round(4))

## 14. Error Analysis and Case Studies

### 14.1 What can and cannot be claimed
The paper's taxonomy (T1-T8) is **not annotated** in SciFact, and the paper's detectors (modality regressor ΔE, PICO scope frames, `delta_num` over aligned quantities) are **not implemented** here. Section 14 therefore uses **transparent lexical/heuristic cues** as *proxies*, applied to real benchmark claims and their retrieved evidence. These cues are noisy (a cue may fire without a real misinterpretation, and real misinterpretations may carry no cue), categories overlap, and subgroup sizes can be small. The tables show *error rates by cue*, which is descriptive evidence, not a validation of the taxonomy.

| Proxy | Taxonomy link **[PAPER]** | Heuristic **[IMPL]** |
|---|---|---|
| hedging erased | T1 / T7 | evidence contains a hedge (*may, might, suggest, possibly, ...*) and the claim does not |
| causal verb in claim | T2 | claim contains *cause(s), lead(s) to, result(s) in, induce, trigger, prevent, drive* |
| scope shift | T3 | claim mentions humans/patients, retrieved evidence mentions only animal/in-vitro systems |
| universal quantifier | T3 / T7 (unsupported generalisation) | claim contains *all, every, always, never, only, completely, ...* |
| contradictory evidence | T4 | gold label is REFUTED |
| number not matched | T5 | nearest-number alignment: relative gap `delta_num` (paper Eq. 13, crude alignment) > tolerance, or the evidence has no numbers |
| study-design mismatch | T2 / T6 ("methodological conflation") | claim names a study design (RCT, cohort, in-vitro, animal, meta-analysis) that the evidence text does not |
| compound claim | T6 | the Section-6 splitter returns ≥ 2 atomic parts |

In [ ]:
HEDGE_RE = re.compile(r"\b(may|might|could|suggest(?:s|ed|ing)?|possibl[ey]|potential(?:ly)?|appear(?:s|ed)?|likely|probabl[ey]|perhaps|seem(?:s|ed)?|indicat(?:e|es|ed|ing)|hypothesi[sz]e[sd]?)\b", re.I)
CAUSAL_RE = re.compile(r"\b(cause[sd]?|causing|lead(?:s|ing)? to|led to|result(?:s|ed|ing)? in|induce[sd]?|inducing|trigger(?:s|ed|ing)?|prevent(?:s|ed|ing)?|drive[sn]?|drives)\b", re.I)
UNIVERSAL_RE = re.compile(r"\b(all|every|always|never|none|completely|entirely|universal(?:ly)?|invariably|exclusively|solely|only)\b", re.I)
HUMAN_RE = re.compile(r"\b(humans?|patients?|people|adults?|children|women|men|individuals|clinical)\b", re.I)
MODEL_ORG_RE = re.compile(r"\b(mice|mouse|murine|rats?|rodents?|in vitro|cell lines?|cultured|zebrafish|transgenic|animal models?)\b", re.I)
DESIGN_RES = {"rct": re.compile(r"\b(randomi[sz]ed|clinical trials?|rct)\b", re.I),
              "observational": re.compile(r"\b(observational|cohort|cross-sectional|case-control|retrospective)\b", re.I),
              "in_vitro": re.compile(r"\b(in vitro|cell lines?|cultured)\b", re.I),
              "animal": re.compile(r"\b(mice|mouse|murine|rats?|rodents?)\b", re.I),
              "meta": re.compile(r"\b(meta-analys[ie]s|systematic review)\b", re.I)}
NUM_RE = re.compile(r"(?<![A-Za-z0-9.])\d+(?:\.\d+)?")

def numeric_alignment(claim, evidence, tol=None):
    """Crude version of paper Eq. 13: align each claim number to the nearest evidence number and take the max relative gap."""
    tol = CONFIG["NUM_TOL"] if tol is None else tol
    cn = [float(x) for x in NUM_RE.findall(claim)]
    if not cn:
        return {"has_numbers": False, "delta_num": float("nan"), "flag": False}
    en = [float(x) for x in NUM_RE.findall(evidence)]
    if not en:
        return {"has_numbers": True, "delta_num": float("nan"), "flag": True}
    deltas = [min(abs(v - e) / max(abs(e), 1e-8) for e in en) for v in cn]
    return {"has_numbers": True, "delta_num": float(max(deltas)), "flag": bool(max(deltas) > tol)}

def design_set(text):
    return {k for k, rx in DESIGN_RES.items() if rx.search(text)}

def cue_flags(claim, evidence, gold_label, n_atomic):
    num = numeric_alignment(claim, evidence)
    cd, ed = design_set(claim), design_set(evidence)
    return {
        "T1/T7 hedging erased": bool(HEDGE_RE.search(evidence)) and not HEDGE_RE.search(claim),
        "T2 causal verb in claim": bool(CAUSAL_RE.search(claim)),
        "T3 scope shift (human claim, animal/in-vitro evidence)": bool(HUMAN_RE.search(claim)) and bool(MODEL_ORG_RE.search(evidence)) and not HUMAN_RE.search(evidence),
        "T3/T7 universal quantifier": bool(UNIVERSAL_RE.search(claim)),
        "T4 contradictory evidence (gold REFUTED)": gold_label == "REFUTED",
        "T5 claim contains numbers": num["has_numbers"],
        "T5 number not matched in evidence": num["flag"],
        "T2/T6 study-design mismatch": bool(cd) and not (cd & ed),
        "T6 compound claim": n_atomic >= 2,
    }

# ----- apply to the REAL test claims, using the final system's evidence and predictions (default configuration) -----
FINAL_SYSTEM = SYS_SYN if NLI_AVAILABLE else SYS_LR
e = pred_entry(FINAL_SYSTEM, "test")
bundles = get_bundles("test", DEFAULT["ret"], DEFAULT["K"], with_nli=False)
rows = []
for rec, b, pr in zip(RECS["test"], bundles, e["pred"]):
    f = cue_flags(rec["claim"], b["evidence"], rec["label"], len(decompose_claim(rec["claim"])))
    rows.append({"claim_id": rec["claim_id"], "claim": rec["claim"], "gold": rec["label"], "pred": LABELS[pr],
                 "error": LABELS[pr] != rec["label"], "evidence": b["evidence"], **f})
ERR_DF = pd.DataFrame(rows)
CUE_COLS = [c for c in ERR_DF.columns if c[:1] == "T"]
ERR_DF["no cue fired"] = ~ERR_DF[[c for c in CUE_COLS if "contains numbers" not in c]].any(axis=1)
CUE_COLS = CUE_COLS + ["no cue fired"]

overall_err = ERR_DF["error"].mean(); total_err = int(ERR_DF["error"].sum())
tab = []
for c in CUE_COLS:
    sub = ERR_DF[ERR_DF[c]]
    if len(sub) == 0:
        continue
    ci = sps.binomtest(int(sub["error"].sum()), len(sub)).proportion_ci(confidence_level=0.95, method="wilson")
    tab.append({"cue category": c, "claims with cue": len(sub), "errors": int(sub["error"].sum()),
                "error rate": round(sub["error"].mean(), 3), "95% Wilson CI": f"[{ci.low:.3f}, {ci.high:.3f}]",
                "share of all errors": round(sub["error"].sum() / max(total_err, 1), 3)})
ERR_TABLE = pd.DataFrame(tab)
print(f"[OBS] Error analysis of '{FINAL_SYSTEM}' on the TEST set (n={len(ERR_DF)}, overall error rate {overall_err:.3f}, {total_err} errors).")
print("Cue categories overlap; small subgroups have wide intervals. Heuristic proxies, NOT validated taxonomy labels.")
display(ERR_TABLE)

plt.figure(figsize=(9, 5))
order = ERR_TABLE.sort_values("error rate")
plt.barh(order["cue category"], order["error rate"], color=sns.color_palette("deep")[0], label="Error rate within cue category")
for i, (r, n) in enumerate(zip(order["error rate"], order["claims with cue"])):
    plt.text(r + 0.01, i, f"n={n}", va="center", fontsize=8)
plt.axvline(overall_err, color="red", linestyle="--", label=f"Overall error rate ({overall_err:.2f})")
plt.xlim(0, 1.05); plt.xlabel("Error rate (test set)"); plt.ylabel("Heuristic cue category (taxonomy proxy)")
plt.title(f"Error rate by taxonomy-proxy cue: {FINAL_SYSTEM}"); plt.legend(loc="lower right")
savefig("fig_error_categories.png")

dist = ERR_DF[ERR_DF["error"]].groupby(["gold", "pred"]).size().rename("errors").reset_index()
print("[OBS] Which confusions make up the errors (gold -> predicted):"); display(dist)

In [ ]:
# ---------- real error examples per cue category: sampled with a fixed seed (NOT hand-picked to flatter or hurt any method) ----------
ex_rng = np.random.default_rng(CONFIG["SEED"])
for c in CUE_COLS:
    errs = ERR_DF[ERR_DF[c] & ERR_DF["error"]]
    if len(errs) == 0:
        continue
    print("=" * 110); print(f"[OBS] {c}: {len(errs)} errors; showing up to 2 sampled at random")
    for i in ex_rng.choice(len(errs), size=min(2, len(errs)), replace=False):
        r = errs.iloc[int(i)]
        print(f"  claim {r.claim_id}: {r.claim}\n    gold={r.gold} | predicted={r.pred}\n    retrieved evidence: {r.evidence[:300]}")

### 14.2 Illustrative constructed examples **[ILLUS]**

Scenarios A-C are the diagnostic scenarios from Sec. IV-B of the paper (claims quoted from it); scenario D is added here. **All evidence passages below are written by hand for illustration; none is benchmark data, none enters any metric.** The table shows what the *implemented* components output for them *at run time*; no outcome is assumed in advance.

In [ ]:
ILLUS = [
    {"id": "A", "taxonomy": "T3/T7 scope extrapolation + caveat erasure",
     "claim": "Inhibiting pathway P38 via compound X clinically reverses cognitive decline in late-stage human patients.",
     "evidence": "In an in vitro neuroblastoma assay and a 14-day transgenic mouse trial (n = 8), inhibiting P38 with compound X improved memory scores. Whether this effect translates to human patients remains untested."},
    {"id": "B", "taxonomy": "T3/T6 unsupported generalisation",
     "claim": "Sparse Mixture-of-Experts transformers consistently outperform dense models across all downstream NLP reasoning and translation benchmarks.",
     "evidence": "Sparse MoE models improved pre-training perplexity and high-resource translation. In the ablation, dense models outperformed the MoE variant on multi-step mathematical reasoning and low-resource translation."},
    {"id": "C", "taxonomy": "T5 numerical distortion",
     "claim": "Nitrogen-doped graphene achieves a 45% enhancement in peak current density.",
     "evidence": "Under alkaline conditions, nitrogen-doped graphene showed a 4.5% increase in peak current density."},
    {"id": "D", "taxonomy": "T2 causal overclaiming",
     "claim": "Coffee consumption causes heart disease.",
     "evidence": "In this observational cohort, coffee consumption was associated with a higher incidence of heart disease."},
]
rows = []
tf = SENT_RET["TF-IDF"].vec
for it in ILLUS:
    num = numeric_alignment(it["claim"], it["evidence"])
    flags = cue_flags(it["claim"], it["evidence"], "", len(decompose_claim(it["claim"])))
    row = {"id": it["id"], "taxonomy (expected)": it["taxonomy"],
           "TF-IDF cosine": round(float(tf.transform([it["claim"]]).multiply(tf.transform([it["evidence"]])).sum()), 3),
           "delta_num": None if np.isnan(num["delta_num"]) else round(num["delta_num"], 2),
           "cues fired": "; ".join(k for k, v in flags.items() if v and k[0] == "T" and "contains numbers" not in k) or "-"}
    if NLI_AVAILABLE:
        sents = [s for s in re.split(r"(?<=[.!?])\s+", it["evidence"]) if s]
        lg = nli_forward([(s, it["claim"]) for s in sents])
        dl, _ = decisive_logits(lg)
        tp = TUNED[(SYS_SYN, DEFAULT["ret"], DEFAULT["K"])]
        rel = float((tf.transform(sents) @ tf.transform([it["claim"]]).T).toarray().max())
        pred, p, reason = synthesize_batch(dl[None, :], np.array([rel]), tp["T"], tp["tau_rel"], tp["tau_conf"])
        row.update({"P(ent)/P(con)/P(neu)": "/".join(f"{x:.2f}" for x in p[0]), "pipeline decision": LABELS[int(pred[0])], "gate": reason[0]})
    rows.append(row)
display(pd.DataFrame(rows))
print("Reading guide: for these claims the *intended* verdict is 'not supported as stated'. Compare it with the run-time output above;"
      " any disagreement is a genuine limitation of the implemented components, not something to be tuned away.")

## 15. Discussion of Findings

### 15.1 Mechanical check of the hypotheses (generated from this run's numbers)
The cell below only restates what the computed tables show and applies the pre-specified reading rule from Section 13. It makes **no claim beyond** the benchmark, the models and the partitions used here.

In [ ]:
def hypothesis_report():
    L = []
    def rs(level, metric, a, b):
        s = RET_STAT[(RET_STAT.level == level) & (RET_STAT.metric == metric) & (RET_STAT.A == a) & (RET_STAT.B == b)]
        return None if s.empty else s.iloc[0]
    r = rs("sentence", "MRR", "BM25", "TF-IDF")
    if r is not None:
        verdict = "point estimate consistent with H1" if r["mean diff A-B"] >= 0 else "point estimate NOT consistent with H1"
        L.append(f"H1 (sentence MRR, BM25 - TF-IDF, n={int(r.n_claims)}): {r['mean diff A-B']:+.4f} [{r.ci_low:+.4f}, {r.ci_high:+.4f}] -> {verdict}; "
                 f"{'CI excludes 0' if r['CI excludes 0'] else 'CI includes 0, so the difference is descriptive'}.")
    else:
        L.append("H1: not evaluable (missing results).")
    for a, b in [("Dense", "BM25"), ("Dense", "TF-IDF")]:
        r = rs("abstract", f"hit@{k_d}", a, b)
        if r is not None:
            L.append(f"H2 (abstract hit@{k_d}, {a} - {b}, n={int(r.n_claims)}): {r['mean diff A-B']:+.4f} [{r.ci_low:+.4f}, {r.ci_high:+.4f}] -> "
                     f"{'CI excludes 0' if r['CI excludes 0'] else 'CI includes 0 (descriptive)'}.")
    if not DENSE_AVAILABLE:
        L.append("H2: dense retriever was not available in this run, so H2 could not be evaluated.")
    if NLI_AVAILABLE:
        s = STAT_SYS[(STAT_SYS["system A"] == SYS_NLI) & (STAT_SYS["system B"] == SYS_RET)]
        n_ref = int((pred_entry(SYS_RET, "test")["pred"] == 1).sum())
        L.append(f"H3 (NLI vs retrieval-only, default config): macro-F1 diff {s.iloc[0]['macro-F1 diff A-B [95% CI]']}; reading: {s.iloc[0]['reading']}. "
                 f"Retrieval-only predicted REFUTED for {n_ref} claims (zero by construction).")
        sub = SYS_DF[(SYS_DF.system == SYS_NLI) & (SYS_DF.split == "test")]
        L.append(f"H4 (zero-shot NLI test macro-F1 across {len(sub)} retriever/K configurations): min {sub.macro_f1.min():.3f}, max {sub.macro_f1.max():.3f}; "
                 "compare the range with the CI widths in Section 12 before reading it as an effect.")
        raw, ts = CAL_TABLE.iloc[0], CAL_TABLE.iloc[1]
        L.append(f"H5 (calibration, test): ECE raw {raw['ECE [95% CI]']} vs temperature-scaled {ts['ECE [95% CI]']}; "
                 "overlapping intervals mean no reliable ECE difference can be claimed.")
    big = ERR_TABLE[ERR_TABLE["claims with cue"] >= 10].sort_values("error rate")
    if len(big) >= 2:
        lo_, hi_ = big.iloc[0], big.iloc[-1]
        L.append(f"H6 (cue categories with >= 10 claims): lowest error rate '{lo_['cue category']}' {lo_['error rate']} {lo_['95% Wilson CI']}; "
                 f"highest '{hi_['cue category']}' {hi_['error rate']} {hi_['95% Wilson CI']}. Overlapping intervals mean the difference is not established.")
    else:
        L.append("H6: fewer than two cue categories have >= 10 claims; not evaluable.")
    return L

HYPOTHESIS_LINES = hypothesis_report()
for line in HYPOTHESIS_LINES:
    print("-", line)

### 15.2 Author's interpretation  *(to be completed after execution)*

> **Replace this cell with your own reading of the numbers produced by *your* run.** Suggested structure (do not copy any wording as a finding before you have verified it against your tables):
>
> 1. **Retrieval:** which retriever found the gold rationale most often, and is the difference larger than the bootstrap CI? Does the abstract-level result differ from the sentence-level result?
> 2. **Verification:** how do TF-IDF+LR, retrieval-only, NLI and NLI+synthesis compare in macro-F1 *per class* (especially REFUTED)? What does the retrieval-only confusion matrix say about similarity vs entailment?
> 3. **Uncertainty:** what do the abstention gates and conformal sets change (coverage, set size, abstain share)?
> 4. **Errors:** which cue categories have more errors, with what sample sizes?
> 5. **Relation to the paper:** which of the paper's *claimed mechanisms* (topical-overlap trap, need for calibrated abstention, scope/number checks) are consistent with, inconsistent with, or untestable by this experiment?

## 16. Limitations

**Benchmark proxy.**
* SciFact is not a dataset of misinterpreted citations. REFUTED claims are largely annotator-written negations and NEI means "no evidence in this abstract". Findings concern claim verification on SciFact, **not** the detection of real-world citation distortion.
* Abstract-only, biomedical-only, English-only evidence. The paper's emphasis on full text, tables, figures and multi-domain scope is not tested.
* The paper's four-way label (including *Partially Supported*) is reduced to three labels.

**Pipeline simplifications.** No discourse window, coreference resolution or LLM-based decomposition; no cross-encoder reranking, ColBERT, section weighting, Epistemic Modality Delta, PICO scope frames or aligned `delta_num`; the severity score (Eq. 16) is not implemented; the "known cited document" setting skips Stage 5 (authenticity / paywall horizon); the abstract-level retrieval is a secondary open-domain check.

**Models.** The NLI model is general-domain (MNLI/SNLI-style) unless fine-tuned; a scientific-domain NLI model was not evaluated. The decisive-sentence aggregation is a simple heuristic that cannot combine evidence across sentences.

**Statistics.** Test partition of a few hundred claims, one training run per model, no repeated cross-validation: intervals reflect test-sampling variability only. Thresholds, temperature and `C` are tuned on a small validation partition and are noisy. Conformal coverage is marginal and assumes exchangeability between calibration and test claims, which may not hold across SciFact's annotation batches. Cue-based error categories are heuristics, overlap, and are not human-validated.

**Reproducibility caveats.** Results depend on library and model versions (recorded in `results/environment.json`), on Hugging Face model revisions at download time, and on GPU vs CPU numerical differences. The dense and NLI components require internet access on first run.

## 17. Conclusion

**What this notebook establishes (by construction, independent of results).** It provides a reproducible, leakage-controlled computational case study that operationalises the *retrieve-then-verify-then-abstain* backbone of CEDAR (Stages 7, 9 and 11-12, in simplified form) on a public benchmark, and it makes the paper's central distinction between *topical similarity* and *propositional entailment* directly measurable.

**What it does not establish.** It does not validate CEDAR as a whole, it does not measure the detection of genuinely misinterpreted citations, and it does not show that any component of the full framework (ΔE, scope divergence, multimodal retrieval, severity score) works.

**Findings of this run** *(complete after executing the notebook, using Sections 12-15 and `results/results_summary.md`)*:
* Retrieval: …
* Verification: …
* Uncertainty handling: …
* Error structure: …

**Future work** follows the paper's own agenda: full-text and table/figure evidence, an annotated corpus of genuinely misinterpreted citations with T1-T8 labels, aligned numeric verification, a modality regressor for ΔE, cross-encoder reranking, and scientific-domain NLI backbones.

## 18. Reproducibility and Execution Instructions

1. **Google Colab:** upload the notebook, set *Runtime → Run all* (GPU optional). Internet access is required for `pip`, the SciFact download and Hugging Face models.
2. **Local:** `python -m venv .venv && source .venv/bin/activate && pip install -r requirements.txt && jupyter notebook`.
3. **Configuration:** all switches are in the `CONFIG` dictionary (Section 0). Set `USE_DENSE=False` / `USE_NLI=False` for a fast sparse-only run; set `RUN_FINETUNE=True` for the optional fine-tuning.
4. **Determinism:** the seed is fixed (`CONFIG["SEED"]`); partitions, bootstraps and sampled examples are seeded. GPU kernels may still introduce tiny numerical differences.
5. **Outputs** are written to `results/` (tables as CSV, figures as PNG, `config.json`, `environment.json`, and an auto-generated `results_summary.md`).
6. **If the SciFact download fails**, follow the manual instructions printed by Section 4; nothing is fabricated.

In [ ]:
# ------------------------------------ persist every table, figure, configuration and the audit trail ------------------------------------
def tuned_to_json(tuned):
    return {f"{k[0]} | {k[1]} | K={k[2]}": {a: float(b) for a, b in v.items()} for k, v in tuned.items()}

SYS_DF = pd.DataFrame(SYS_ROWS)
SYS_DF.to_csv(RESULTS_DIR / "system_results_all_configs.csv", index=False)
RET_RESULTS.to_csv(RESULTS_DIR / "retrieval_results.csv", index=False)
RET_STAT.to_csv(RESULTS_DIR / "retrieval_paired_differences.csv", index=False)
MAIN_TABLE.to_csv(RESULTS_DIR / "main_table_test_default_config.csv", index=False)
STAT_SYS.to_csv(RESULTS_DIR / "system_comparisons_mcnemar_holm.csv", index=False)
ERR_TABLE.to_csv(RESULTS_DIR / "error_analysis_by_cue.csv", index=False)
if len(CAL_TABLE): CAL_TABLE.to_csv(RESULTS_DIR / "calibration_test.csv", index=False)
if len(CONF_TABLE): CONF_TABLE.to_csv(RESULTS_DIR / "conformal_test.csv", index=False)

# per-claim predictions of all systems at the default configuration (test) for auditing
audit = pd.DataFrame({"claim_id": pred_entry(SYS_MAJ, "test")["claim_ids"], "gold": [LABELS[i] for i in pred_entry(SYS_MAJ, "test")["y"]]})
for s in available_systems():
    audit[s] = [LABELS[i] for i in pred_entry(s, "test")["pred"]]
audit.to_csv(RESULTS_DIR / "test_predictions_default_config.csv", index=False)
ERR_DF.drop(columns=["evidence"]).to_csv(RESULTS_DIR / "test_error_analysis_per_claim.csv", index=False)

with open(RESULTS_DIR / "config.json", "w") as f:
    json.dump({"CONFIG": CONFIG, "DEFAULT_CONFIGURATION": DEFAULT, "TUNED_PARAMETERS": tuned_to_json(TUNED),
               "SPLIT_SIZES": {s: len(v) for s, v in RECS.items()}, "DENSE_AVAILABLE": DENSE_AVAILABLE, "NLI_AVAILABLE": NLI_AVAILABLE}, f, indent=2, default=str)
with open(RESULTS_DIR / "environment.json", "w") as f:
    json.dump(ENVIRONMENT, f, indent=2)

# ------------------------------------ auto-generated summary (numbers only come from this run) ------------------------------------
def md_block(df):
    return "```\n" + df.to_string(index=False) + "\n```\n"

summary = [f"# Results summary (auto-generated)\n",
           f"Generated: {pd.Timestamp.now():%Y-%m-%d %H:%M} | seed {CONFIG['SEED']} | NLI model: "
           f"{CONFIG['NLI_MODEL'] if NLI_AVAILABLE else 'not used'} | dense model: {CONFIG['DENSE_MODEL'] if DENSE_AVAILABLE else 'not used'}\n",
           "These numbers were produced by executing the notebook; they are not edited by hand. They describe a computational case study "
           "on SciFact (a proxy benchmark), **not** a validation of the full CEDAR framework.\n",
           f"## Data\nClaims per partition: {({s: len(v) for s, v in RECS.items()})}. Test = official SciFact dev. "
           f"Default configuration (validation-selected): {DEFAULT}.\n",
           "## Retrieval (test; value [95% cluster-bootstrap CI])\n", md_block(RET_RESULTS[RET_RESULTS.split == 'test'].assign(
               value=lambda d: d.value.round(3), ci_low=lambda d: d.ci_low.round(3), ci_high=lambda d: d.ci_high.round(3))),
           "## Verification (test, default configuration)\n", md_block(MAIN_TABLE),
           "## Pre-specified system comparisons\n", md_block(STAT_SYS)]
if len(CAL_TABLE):
    summary += ["## Calibration\n", md_block(CAL_TABLE), "## Split conformal sets\n", md_block(CONF_TABLE)]
summary += ["## Error analysis by heuristic cue (proxies, not validated taxonomy labels)\n", md_block(ERR_TABLE),
            "## Mechanical hypothesis check\n", "\n".join(f"- {l}" for l in HYPOTHESIS_LINES) + "\n",
            "## Limitations\nSciFact is a proxy (abstract-only, biomedical, annotator-written refutations); many CEDAR components are not implemented; "
            "single test partition; tuned thresholds are noisy; cue-based categories are heuristic; conformal guarantees are marginal and assume exchangeability.\n"]
(RESULTS_DIR / "results_summary.md").write_text("\n".join(summary), encoding="utf-8")

print("Saved to", RESULTS_DIR.resolve())
for p in sorted(RESULTS_DIR.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(RESULTS_DIR))
# In Colab, to download everything:  !zip -r results.zip results  then  from google.colab import files; files.download("results.zip")